# Bài 17 · Transformer & GPT: tự xây một GPT nhỏ viết tiếp Truyện Kiều, rồi soi vào GPT-2 thật

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/17-transformer.ipynb)

Notebook đi kèm [Bài 17 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/) (reel và demo tương tác nằm ở trang bài học).
Ta tự viết bằng numpy một GPT đúng kiểu GPT-2: embedding token và vị trí, tự chú ý có mặt nạ nhiều đầu, khối pre-LN
có kết nối tắt và MLP, softmax ở đầu ra, lượt ngược viết tay. Ta huấn luyện nó ngay trong notebook trên Truyện Kiều,
từng ký tự một, rồi đặt nó cạnh các số đo trên GPT-2 small thật (124 triệu tham số) mà demo dùng.

**Bạn sẽ làm:**
1. Biến ký tự thành vectơ: embedding token cộng embedding vị trí.
2. Viết tự chú ý có mặt nạ (vòng lặp trước, ma trận sau), nhiều đầu, cả khối Transformer, và kiểm lượt ngược bằng
   sai phân hữu hạn.
3. Huấn luyện GPT nhỏ bằng AdamW, kiểm rằng mặt nạ thật sự che tương lai, rồi sinh thơ với nhiệt độ và top-k.
4. Viết bộ nhớ đệm KV, kiểm rằng nó cho đúng kết quả của lượt đầy đủ, rồi đếm phép tính và bộ nhớ của GPT-2.
5. Nuôi một đầu quy nạp trong mô hình 2 khối, rồi đọc các đầu, logit lens và phân phối 50.257 token của GPT-2 thật.

**Cần biết trước:** [Bài 14 · Token & embedding](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/),
[Bài 16 · Cơ chế chú ý](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/attention/),
[Bài 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chuan-hoa/) và
[Bài 13 · Kết nối tắt & ResNet](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/resnet/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải dữ liệu (~0,4 MB) lần đầu.

Mọi con số về GPT nhỏ đều do chính các ô bên dưới tính ra. Số đo trên GPT-2 đọc từ đúng tệp kết quả mà demo dùng
(notebook không tải 548 MB trọng số của GPT-2); riêng số tham số, phép tính và bộ nhớ của GPT-2 thì ta tự tính lại từ
cấu hình đã công bố, và chúng trùng với demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import hashlib
import json
import math
import re
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
SEQ = LinearSegmentedColormap.from_list("trang_xanh", ["#f7f7f7", BLUE, "#0b2a59"])  # 0 → lớn (trọng số chú ý)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def sci(x):
    """Số rất nhỏ kiểu Việt Nam: 2,4·10⁻⁷ (0 in là 0)."""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)))
    return f"{vn(x / 10**e, 1)}·10{str(e).translate(SUP)}"


def show(M, digits=3):
    """In một ma trận nhỏ với dấu phẩy thập phân."""
    print("\n".join("  ".join(f"{vn(v, digits):>6}" for v in row) for row in M))


print(vn(124439808, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.446), sci(2.4e-7))

## 2 · Dữ liệu: Truyện Kiều, từng ký tự

Ta dùng *Truyện Kiều* của Nguyễn Du (đầu thế kỷ 19), một tác phẩm thuộc phạm vi công cộng. Bản chép lấy từ
[Wikisource tiếng Việt](https://vi.wikisource.org/w/index.php?oldid=205132), ghim ở phiên bản 205132 và kiểm mã MD5,
giống hệt Bài 15, nên ai chạy cũng có đúng cùng một văn bản. Wikimedia từ chối các yêu cầu không có User-Agent, nên ô
dưới đặt một User-Agent trước khi tải.

Ô cũng nạp hai tệp kết quả mà demo dùng (`fetch_json`, có kiểm tra MD5): các số đo trên GPT-2 small thật, tính một lần
bằng numpy trên trọng số của mô hình. Ta đọc chúng ở mục 12.

In [ ]:
SRC = "https://vi.wikisource.org/w/index.php?oldid=205132&action=raw"  # Truyện Kiều, phiên bản 205132
RAW_MD5 = "d4090daac094841888ee2b2b18661065"
opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "HocSauReels-notebook/1.0 (education; https://d3qr1cjln6qqtk.cloudfront.net/)")]
urllib.request.install_opener(opener)  # mọi lần tải bên dưới đều gửi User-Agent này
RAW = Path("truyen_kieu_205132.wiki")
if not RAW.exists():
    urllib.request.urlretrieve(SRC, RAW)
assert hashlib.md5(RAW.read_bytes()).hexdigest() == RAW_MD5, f"{RAW} bị hỏng, hãy xóa và tải lại"


# Kết quả đo trên GPT-2 thật (cùng tệp với demo trên web).
# Khi xuất bản, build.py thay chỗ giữ chỗ COURSE_DATA_URL bên dưới bằng thư mục data/ của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


inside = fetch_json("gpt2-inside.json", "9ee32934381ae700b3d18a63059a20b6")  # dự đoán, logit lens, các đầu
samp = fetch_json("gpt2-sampling.json", "20257a3d7e84893538a9851f5712b268")  # phân phối token kế tiếp
size = sum(f.stat().st_size for f in (RAW, Path("data/gpt2-inside.json"), Path("data/gpt2-sampling.json")))
print(f"đã có 3 tệp, tổng {vn(size / 1e6, 1)} MB")

Tệp tải về là mã nguồn wiki của trang. Ta lấy phần giữa hai thẻ `<poem>`, bỏ các số thứ tự câu `{{số|…}}` và các dòng
trống, giống Bài 15. Từ vựng là mọi ký tự xuất hiện trong văn bản; mỗi ký tự là một token, mã số của nó là thứ tự
trong danh sách đã sắp xếp. Như Bài 15, 3.000 câu thơ đầu để huấn luyện, 254 câu cuối để kiểm định.

In [ ]:
body = RAW.read_bytes().decode("utf-8").split("<poem>")[1].split("</poem>")[0]
lines = [re.sub(r"\{\{số\|\d+\}\}", "", ln).strip() for ln in body.split("\n")]  # bỏ số thứ tự câu
text = "\n".join(ln for ln in lines if ln) + "\n"
assert hashlib.md5(text.encode()).hexdigest() == "dd94ee390bfc543287b11af62b2f2fc8"
LINES = text[:-1].split("\n")
CHARS = sorted(set(text))  # từ vựng: mọi ký tự khác nhau, kể cả dấu cách và "\n"
V = len(CHARS)
stoi = {ch: i for i, ch in enumerate(CHARS)}
ids = np.array([stoi[ch] for ch in text])
cut = len("\n".join(LINES[:3000])) + 1
train_ids, val_ids = ids[:cut], ids[cut:]  # 3.000 câu đầu / 254 câu cuối


def decode(seq):
    return "".join(CHARS[i] for i in seq)


print(f"{vn(len(LINES), 0)} câu thơ, {vn(len(text), 0)} ký tự, từ vựng {V} ký tự")
print(
    f"huấn luyện: 3.000 câu đầu, {vn(len(train_ids), 0)} ký tự · kiểm định: {len(LINES) - 3000} câu cuối, {vn(len(val_ids), 0)} ký tự"
)
print("mã số của 'T', 'r', 'ă', 'm':", [stoi[c] for c in "Trăm"])
print("\n".join(LINES[:4]))
assert (len(LINES), len(text), V, len(train_ids), len(val_ids)) == (3254, 104204, 132, 96204, 8000)

Mô hình ngôn ngữ chỉ học một việc: đọc các ký tự đã có, đoán ký tự kế tiếp. Trước khi xây Transformer, ta đặt hai mốc
để so. Đoán đều trên 132 ký tự tốn $\log_2 132$ bit cho mỗi ký tự. Mô hình **bigram** chỉ nhìn đúng ký tự liền trước:
đếm các cặp trong tập huấn luyện, cộng thêm một chút $\alpha$ để không cặp nào có xác suất 0. Thước đo là
**bit mỗi ký tự** (entropy chéo cơ số 2, Bài 04) trên tập kiểm định: càng thấp càng tốt.

In [ ]:
uniform_bits = math.log2(V)
counts = np.zeros((V, V))
np.add.at(counts, (train_ids[:-1], train_ids[1:]), 1)  # counts[a, b] = số lần b đứng ngay sau a
for alpha in (0.01, 0.1, 1.0):
    P_bi = (counts + alpha) / (counts + alpha).sum(1, keepdims=True)
    print(f"bigram, α = {vn(alpha, 2)}: {vn(-np.log2(P_bi[val_ids[:-1], val_ids[1:]]).mean(), 3)} bit/ký tự")
P_bigram = (counts + 0.01) / (counts + 0.01).sum(1, keepdims=True)
bigram_bits = -np.log2(P_bigram[val_ids[:-1], val_ids[1:]]).mean()
print(f"đoán đều: log₂ {V} = {vn(uniform_bits, 3)} bit/ký tự")

Bigram tốt nhất ở $\alpha = 0{,}01$: 3,14 bit, tốt hơn nhiều so với 7,04 bit của đoán đều. Nhưng nó không nhớ gì ngoài
một ký tự, nên không biết mình đang ở giữa một chữ, cuối câu 6 hay câu 8. Muốn dùng cả đoạn đứng trước, mỗi vị trí phải
*lấy thông tin từ các vị trí trước nó*. Đó là việc của tự chú ý.

## 3 · Đầu vào: embedding token cộng embedding vị trí

Như Bài 14, mỗi token tra ra một hàng của bảng embedding $E$ (cỡ $V \times d$). GPT còn có bảng thứ hai $P$, mỗi
**vị trí** một vectơ học được, và cộng hai vectơ lại:

$$h_0[t] = E[x_t] + P[t]$$

GPT nhỏ của ta có $d = 48$ và một khung ngữ cảnh 64 ký tự; GPT-2 small có $d = 768$, khung 1.024 token và 50.257 token
trong từ vựng. Tự chú ý *không có mặt nạ* hoàn toàn không biết thứ tự: hoán vị các token đầu vào thì kết quả chỉ bị hoán
vị theo (ở Bài 16, bộ đọc không có vị trí trả lời 23 cho mọi thứ tự của “hai mươi ba”). Có $P$, cùng chữ “a” ở vị trí 3
và vị trí 40 là hai vectơ khác nhau. (Mặt nạ nhân quả ở mục 4 tự nó cũng để lộ một ít thông tin vị trí: Haviv và cộng sự,
2022, huấn luyện được GPT không có $P$. GPT-2 vẫn dùng $P$.)

Khởi tạo: $E \sim \mathcal N(0, 1)$, $P \sim \mathcal N(0;\, 0{,}1^2)$, mỗi ma trận $\mathcal N(0, 1/n_\text{vào})$, như
mô hình của Bài 23. Hai ma trận ghi vào dòng dư ($W_O$, $W_2$) nhân thêm $1/\sqrt{2L}$, theo bài báo GPT-2 (nhân
$1/\sqrt N$ với $N$ lớp dư), để tổng của nhiều khối không phình ra. Ma trận đầu ra bắt đầu bằng 0.

In [ ]:
CFG = {"V": V, "ctx": 64, "d": 48, "L": 2, "heads": 4}  # từ vựng, khung ngữ cảnh, độ rộng, số khối, số đầu


def init_gpt(cfg, seed=0, dtype=np.float32):
    rng = np.random.default_rng(seed)
    d, L = cfg["d"], cfg["L"]
    r = 1 / math.sqrt(2 * L)

    def mat(n_in, n_out, scale=1.0):
        return rng.normal(0, scale / math.sqrt(n_in), (n_in, n_out)).astype(dtype)

    def vec(n, value=0.0):
        return np.full(n, value, dtype)

    p = {"wte": rng.normal(0, 1, (cfg["V"], d)).astype(dtype), "wpe": rng.normal(0, 0.1, (cfg["ctx"], d)).astype(dtype)}
    for l in range(L):
        p |= {f"{l}.g1": vec(d, 1), f"{l}.b1": vec(d)}  # LayerNorm trước lớp chú ý
        p |= {f"{l}.Wqkv": mat(d, 3 * d), f"{l}.bqkv": vec(3 * d)}  # Q, K, V của mọi đầu, gộp một ma trận
        p |= {f"{l}.Wo": mat(d, d, r), f"{l}.bo": vec(d)}  # W_O: nối các đầu rồi chiếu về dòng dư
        p |= {f"{l}.g2": vec(d, 1), f"{l}.b2": vec(d)}  # LayerNorm trước MLP
        p |= {f"{l}.W1": mat(d, 4 * d), f"{l}.c1": vec(4 * d)}  # MLP d → 4d
        p |= {f"{l}.W2": mat(4 * d, d, r), f"{l}.c2": vec(d)}  # 4d → d
    p |= {"gf": vec(d, 1), "bf": vec(d)}  # LayerNorm cuối
    p |= {"Wh": np.zeros((d, cfg["V"]), dtype), "bh": vec(cfg["V"])}  # đầu ra: d → V logit
    return p


p0 = init_gpt(CFG)
window = train_ids[: CFG["ctx"]]
h0 = p0["wte"][window] + p0["wpe"][: len(window)]  # tra bảng rồi cộng
print("một khung:", repr(decode(window[:36])), "…")
print("h₀:", h0.shape, "· E:", p0["wte"].shape, "· P:", p0["wpe"].shape)

## 4 · Tự chú ý có mặt nạ: một đầu

Như Bài 16, mỗi vị trí $t$ tạo ba vectơ hàng từ $h[t]$: truy vấn $q_t = h[t]W_Q$ (“tôi cần gì?”), khóa $k_t = h[t]W_K$
(“tôi có gì?”) và giá trị $v_t = h[t]W_V$ (“tôi đưa gì nếu được chọn?”). Vị trí $t$ chấm điểm mọi vị trí $j$, chia
$\sqrt{d_h}$ ($d_h$ là cỡ một đầu), softmax, rồi lấy trung bình có trọng số của các $v_j$:

$$s_{tj} = \frac{q_t \cdot k_j}{\sqrt{d_h}}, \qquad \alpha_{tj} = \operatorname{softmax}_j(s_{tj}), \qquad
o_t = \sum_j \alpha_{tj}\, v_j$$

Điểm khác Bài 16: đây là **tự** chú ý (truy vấn và khóa cùng đến từ một chuỗi) và có **mặt nạ nhân quả**: vị trí $t$ chỉ
được nhìn $j \le t$. Ô dưới viết phép tính hai lần: vòng lặp theo đúng định nghĩa, rồi dạng ma trận
$\operatorname{softmax}(QK^\top/\sqrt{d_h} + M)\,V$, trong đó $M$ gán $-\infty$ cho mọi ô $j > t$ (vì $e^{-\infty} = 0$).
Ta lấy đầu thứ nhất của khối 0 và bỏ qua LayerNorm cho gọn.

In [ ]:
def softmax(s):
    e = np.exp(s - s.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


def attention_loop(q, k, v):
    """Một đầu, từng vị trí: vị trí t chấm điểm j = 0…t, softmax, cộng các v_j theo trọng số."""
    n, dh = q.shape
    out = np.zeros_like(v)
    for t in range(n):
        s = np.array([q[t] @ k[j] for j in range(t + 1)]) / math.sqrt(dh)
        alpha = softmax(s)
        out[t] = sum(alpha[j] * v[j] for j in range(t + 1))
    return out


def attention(q, k, v, causal=True):
    """Cùng phép tính dạng ma trận; q, k, v có thể có thêm các trục đầu (…, n, d_h)."""
    n, dh = q.shape[-2:]
    s = q @ np.swapaxes(k, -1, -2) / math.sqrt(dh)
    if causal:
        s = np.where(np.triu(np.ones((n, n), bool), 1), -np.inf, s)  # ô j > t: −∞
    A = softmax(s)
    return A @ v, A


d, nh = CFG["d"], CFG["heads"]
dh = d // nh
qkv = h0 @ p0["0.Wqkv"]  # (64, 3·48): Q, K, V của cả 4 đầu, cạnh nhau
q1, k1, v1 = qkv[:, :dh], qkv[:, d : d + dh], qkv[:, 2 * d : 2 * d + dh]  # đầu thứ nhất: 12 cột mỗi loại
o_loop = attention_loop(q1, k1, v1)
o_mat, A = attention(q1, k1, v1)
print(f"q, k, v mỗi cái {q1.shape}; vòng lặp và ma trận lệch nhau tối đa {sci(np.abs(o_loop - o_mat).max())}")
print("5 hàng đầu của ma trận chú ý A (hàng = vị trí truy vấn t, cột = vị trí j):")
show(A[:5, :5])
assert np.allclose(o_loop, o_mat, atol=1e-5) and np.all(np.triu(A, 1) == 0)

Phía trên đường chéo của $A$ toàn số 0 *đúng bằng 0*, và mỗi hàng cộng lại bằng 1. Hàng 0 chỉ có một ô: vị trí đầu tiên
chỉ nhìn được chính nó.

Một trường hợp đặc biệt giúp thấy mặt nạ làm gì: nếu mọi điểm bằng nhau (ví dụ $q = 0$), softmax chia đều, và
$o_t$ là **trung bình** của $v_0, \dots, v_t$. Ma trận $A$ khi đó là ma trận “trung bình cộng dồn” mà Karpathy dùng để
mở đầu bài *Let's build GPT*:

In [ ]:
_, A_uniform = attention(np.zeros_like(q1[:4]), k1[:4], v1[:4])
show(A_uniform)

## 5 · Nhiều đầu, rồi cả khối Transformer

**Nhiều đầu.** Thay vì một phép chú ý cỡ $d$, ta chia $d = 48$ thành 4 đầu, mỗi đầu $d_h = 12$ chiều, mỗi đầu có $W_Q$,
$W_K$, $W_V$ riêng và tự chọn nhìn vào đâu. Trong code, $Q$, $K$, $V$ của mọi đầu nằm cạnh nhau trong một ma trận
$W_{QKV}$ cỡ $d \times 3d$ (như GPT-2), nên chỉ cần một phép nhân rồi đổi hình mảng. Kết quả của các đầu được **nối**
lại thành vectơ $d$ chiều rồi nhân với $W_O$. GPT-2 small làm đúng như vậy với 12 đầu × 64 chiều = 768.

Ô dưới chia $h_0 W_{QKV}$ thành 4 đầu và kiểm rằng đầu thứ nhất chính là đầu ta vừa tính bằng tay ở mục 4.

In [ ]:
n = len(window)
q, k, v = qkv.reshape(n, 3, nh, dh).transpose(1, 2, 0, 3)  # mỗi cái (đầu, vị trí, d_h) = (4, 64, 12)
o_heads, A_heads = attention(q, k, v)  # cả 4 đầu trong một lần gọi
o_cat = o_heads.transpose(1, 0, 2).reshape(n, d)  # nối 4 đầu: (64, 48)
out = o_cat @ p0["0.Wo"]  # W_O trộn các đầu, về lại d chiều
print("q:", q.shape, "→ nối các đầu:", o_cat.shape, "→ sau W_O:", out.shape)
assert np.allclose(o_heads[0], o_mat)  # đầu 0 trùng với phép tính một đầu ở trên

**Khối Transformer.** GPT-2 xếp chồng $L$ khối giống nhau. Mỗi khối có hai lớp con, mỗi lớp con được *cộng vào* vectơ
của token qua kết nối tắt (Bài 13), và LayerNorm (Bài 10) đặt ở **đầu vào** của lớp con (pre-LN; Transformer gốc đặt
sau phép cộng):

$$h \leftarrow h + \operatorname{MSA}(\operatorname{LN}_1(h)), \qquad
h \leftarrow h + \operatorname{GELU}(\operatorname{LN}_2(h)\,W_1 + c_1)\,W_2 + c_2$$

Lớp chú ý là nơi các vị trí *trao đổi* thông tin; MLP ($d \to 4d \to d$, hàm kích hoạt GELU dạng $\tanh$ của GPT-2) xử
lý **riêng từng vị trí**. Vectơ $h$ chạy dọc các khối được gọi là **dòng dư** (residual stream): mỗi lớp con đọc nó và
cộng thêm một phần. Cuối cùng là một LayerNorm nữa rồi lớp đầu ra $z = hW_h + b_h$ cho 132 logit. (GPT-2 không có $W_h$
riêng mà dùng lại $E^\top$; ta giữ $W_h$ riêng như Llama, để lớp đầu ra học tự do hơn.)

Ô dưới viết cả lượt xuôi. Nó nhận một lô $X$ cỡ (B, n) và trả về logit cỡ (B, n, V): **một dự đoán ở mỗi vị trí**.
Nó lưu mọi giá trị trung gian, vì lượt ngược sẽ cần chúng.

In [ ]:
C_GELU = math.sqrt(2 / math.pi)


def layernorm(x, g, b, eps=1e-5):
    xc = x - x.mean(-1, keepdims=True)
    inv = 1 / np.sqrt((xc * xc).mean(-1, keepdims=True) + eps)
    return xc * inv * g + b, (xc * inv, inv)


def gelu(u):
    """GELU dạng tanh, như GPT-2."""
    t = np.tanh(C_GELU * (u + 0.044715 * u * u * u))
    return 0.5 * u * (1 + t), t


def forward(p, X, cfg, causal=True):
    B, n = X.shape
    d, nh = cfg["d"], cfg["heads"]
    h = p["wte"][X] + p["wpe"][:n]  # mục 3
    layers = []
    for l in range(cfg["L"]):
        a, ln1 = layernorm(h, p[f"{l}.g1"], p[f"{l}.b1"])
        qkv = a @ p[f"{l}.Wqkv"] + p[f"{l}.bqkv"]
        q, k, v = qkv.reshape(B, n, 3, nh, d // nh).transpose(2, 0, 3, 1, 4)  # mỗi cái (B, đầu, n, d_h)
        o, A = attention(q, k, v, causal)  # mục 4, mọi đầu cùng lúc
        o = o.transpose(0, 2, 1, 3).reshape(B, n, d)  # nối các đầu
        h = h + o @ p[f"{l}.Wo"] + p[f"{l}.bo"]  # kết nối tắt quanh lớp chú ý
        m, ln2 = layernorm(h, p[f"{l}.g2"], p[f"{l}.b2"])
        u = m @ p[f"{l}.W1"] + p[f"{l}.c1"]
        gu, t = gelu(u)
        h = h + gu @ p[f"{l}.W2"] + p[f"{l}.c2"]  # kết nối tắt quanh MLP
        layers.append({"a": a, "ln1": ln1, "q": q, "k": k, "v": v, "A": A, "o": o, "m": m, "ln2": ln2, "u": u})
        layers[-1] |= {"gu": gu, "t": t}
    hf, lnf = layernorm(h, p["gf"], p["bf"])
    return hf @ p["Wh"] + p["bh"], {"X": X, "layers": layers, "hf": hf, "lnf": lnf}


def cross_entropy(z, Y, w=None):
    """Mất mát trung bình (nat) trên các vị trí, có thể đánh trọng số w, và gradient ∂L/∂z."""
    z2, y = z.reshape(-1, z.shape[-1]), Y.ravel()
    w = np.ones(len(y), z.dtype) if w is None else w.ravel().astype(z.dtype)
    P = softmax(z2)
    loss = -(w * np.log(P[np.arange(len(y)), y])).sum() / w.sum()
    P[np.arange(len(y)), y] -= 1
    return float(loss), (P * (w / w.sum())[:, None]).reshape(z.shape)

Đếm tham số của GPT nhỏ: mỗi khối có $W_{QKV}$ ($3d^2$), $W_O$ ($d^2$), $W_1$ và $W_2$ ($4d^2$ mỗi cái), tức $12d^2$, cộng
$13d$ độ lệch và hệ số LayerNorm. Ô dưới đếm hai cách và kiểm rằng chúng khớp. Nó cũng chạy lượt xuôi lần đầu.

### 🤔 Dự đoán trước
Trước khi học, mất mát trên một lô là bao nhiêu bit mỗi ký tự? (Gợi ý: ma trận đầu ra $W_h$ bắt đầu bằng 0.)

In [ ]:
L_, V_ = CFG["L"], CFG["V"]
formula = L_ * (12 * d * d + 13 * d) + 2 * d + V_ * d + CFG["ctx"] * d + d * V_ + V_
n_params = sum(a.size for a in p0.values())
print(f"{len(p0)} mảng tham số, tổng {vn(n_params, 0)} số; công thức cho {vn(formula, 0)}")
X0 = np.stack([train_ids[i : i + 64] for i in (0, 1000, 2000, 3000)])  # 4 khung, mỗi khung 64 ký tự
Y0 = np.stack([train_ids[i + 1 : i + 65] for i in (0, 1000, 2000, 3000)])  # đáp án: ký tự kế tiếp ở mỗi vị trí
z0, _ = forward(p0, X0, CFG)
loss0, _ = cross_entropy(z0, Y0)
print(f"logit: {z0.shape} · mất mát ban đầu {vn(loss0 / math.log(2), 4)} bit/ký tự; log₂ {V} = {vn(uniform_bits, 4)}")
assert n_params == formula == 72516 and abs(loss0 - math.log(V)) < 1e-5

Đúng $\log_2 132$: với $W_h = 0$ và $b_h = 0$ mọi logit bằng 0, softmax chia đều, nên mô hình mới khởi tạo là “đoán
đều” dù 72.516 tham số kia ngẫu nhiên thế nào. Khởi tạo lớp đầu ra bằng 0 là một lựa chọn có chủ ý: mất mát ban đầu
đúng bằng mốc đoán đều, không cao hơn.

## 6 · Lượt ngược viết tay, kiểm bằng sai phân hữu hạn

Lượt ngược đi qua đúng các bước của lượt xuôi theo chiều ngược lại (Bài 06). Có ba chỗ mới so với các bài trước:
- **Softmax của chú ý:** với $\alpha = \operatorname{softmax}(s)$ theo từng hàng,
  $\partial L/\partial s = \alpha \odot (\partial L/\partial \alpha - \sum_j \alpha_j\, \partial L/\partial \alpha_j)$.
  Ô bị che có $\alpha = 0$, nên tự động nhận gradient 0: mặt nạ không cần xử lý riêng.
- **Kết nối tắt:** $h + F(h)$ có đạo hàm $1 + F'$, nên gradient của dòng dư đi thẳng qua và được *cộng* thêm phần qua
  lớp con (Bài 13).
- **Embedding:** gradient của $E$ cộng dồn vào hàng của token đã dùng (`np.add.at`, vì một ký tự xuất hiện nhiều lần).

In [ ]:
def layernorm_back(dy, g, cache):
    xh, inv = cache
    dxh = dy * g
    dx = inv * (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True))
    return dx, (dy * xh).reshape(-1, xh.shape[-1]).sum(0), dy.reshape(-1, xh.shape[-1]).sum(0)


def gelu_back(dy, u, t):
    return dy * (0.5 * (1 + t) + 0.5 * u * (1 - t * t) * C_GELU * (1 + 3 * 0.044715 * u * u))


def flat(a):
    return a.reshape(-1, a.shape[-1])

In [ ]:
def backward(p, cache, dz, cfg):
    X, hf = cache["X"], cache["hf"]
    B, n = X.shape
    d, nh = cfg["d"], cfg["heads"]
    dh = d // nh
    g = {"Wh": flat(hf).T @ flat(dz), "bh": flat(dz).sum(0)}
    dr, g["gf"], g["bf"] = layernorm_back(dz @ p["Wh"].T, p["gf"], cache["lnf"])  # dr = ∂L/∂(dòng dư)
    for l in reversed(range(cfg["L"])):
        c = cache["layers"][l]
        g[f"{l}.W2"], g[f"{l}.c2"] = flat(c["gu"]).T @ flat(dr), flat(dr).sum(0)  # MLP
        du = gelu_back(dr @ p[f"{l}.W2"].T, c["u"], c["t"])
        g[f"{l}.W1"], g[f"{l}.c1"] = flat(c["m"]).T @ flat(du), flat(du).sum(0)
        dx, g[f"{l}.g2"], g[f"{l}.b2"] = layernorm_back(du @ p[f"{l}.W1"].T, p[f"{l}.g2"], c["ln2"])
        dr = dr + dx  # kết nối tắt: đi thẳng + đi qua MLP
        g[f"{l}.Wo"], g[f"{l}.bo"] = flat(c["o"]).T @ flat(dr), flat(dr).sum(0)  # chú ý
        do = (dr @ p[f"{l}.Wo"].T).reshape(B, n, nh, dh).transpose(0, 2, 1, 3)
        A, q, k, v = c["A"], c["q"], c["k"], c["v"]
        dA, dv = do @ v.transpose(0, 1, 3, 2), A.transpose(0, 1, 3, 2) @ do
        ds = A * (dA - (dA * A).sum(-1, keepdims=True)) / math.sqrt(dh)  # qua softmax; ô bị che: A = 0
        dq, dk = ds @ k, ds.transpose(0, 1, 3, 2) @ q
        dqkv = np.stack([dq, dk, dv]).transpose(1, 3, 0, 2, 4).reshape(B, n, 3 * d)
        g[f"{l}.Wqkv"], g[f"{l}.bqkv"] = flat(c["a"]).T @ flat(dqkv), flat(dqkv).sum(0)
        dx, g[f"{l}.g1"], g[f"{l}.b1"] = layernorm_back(dqkv @ p[f"{l}.Wqkv"].T, p[f"{l}.g1"], c["ln1"])
        dr = dr + dx
    g["wpe"] = np.zeros_like(p["wpe"])
    g["wpe"][:n] = dr.sum(0)
    g["wte"] = np.zeros_like(p["wte"])
    np.add.at(g["wte"], X.ravel(), flat(dr))
    return g

Kiểm từng mảng tham số bằng sai phân trung tâm $\frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}$, như
Bài 06: một GPT tí hon (từ vựng 7, khung 5, $d = 8$, 2 khối, 2 đầu) bằng float64, các tham số được xáo thêm nhiễu để
không mảng nào nằm ở giá trị đặc biệt (LayerNorm có $g = 1$, $W_h = 0$), và mất mát có trọng số ngẫu nhiên $w$.

Ô cũng in gradient của độ lệch khóa $b_K$ (một phần của $b_{QKV}$). Hãy đoán trước: nó lớn cỡ nào?

In [ ]:
cfg_t = {"V": 7, "ctx": 5, "d": 8, "L": 2, "heads": 2}
rng = np.random.default_rng(0)
pt = {k_: a + 0.3 * rng.standard_normal(a.shape) for k_, a in init_gpt(cfg_t, 1, np.float64).items()}
Xt, Yt, wt = rng.integers(0, 7, (3, 5)), rng.integers(0, 7, (3, 5)), rng.random((3, 5))
zt, ct = forward(pt, Xt, cfg_t)
gt = backward(pt, ct, cross_entropy(zt, Yt, wt)[1], cfg_t)
worst, eps = 0.0, 1e-5
for name, a in pt.items():
    for _ in range(4):  # 4 phần tử ngẫu nhiên của mỗi mảng
        i = tuple(rng.integers(0, s) for s in a.shape)
        old = a[i]
        a[i] = old + eps
        lp = cross_entropy(forward(pt, Xt, cfg_t)[0], Yt, wt)[0]
        a[i] = old - eps
        lm = cross_entropy(forward(pt, Xt, cfg_t)[0], Yt, wt)[0]
        a[i] = old
        num = (lp - lm) / (2 * eps)
        worst = max(worst, abs(num - gt[name][i]) / max(1e-4, abs(num) + abs(gt[name][i])))
print(f"{len(pt)} mảng tham số × 4 phần tử: sai số tương đối lớn nhất {sci(worst)}")
bk = np.abs(gt["0.bqkv"][cfg_t["d"] : 2 * cfg_t["d"]]).max()  # phần độ lệch của khóa K
print(f"gradient lớn nhất của độ lệch khóa b_K: {sci(bk)}")
assert worst < 1e-6 and bk < 1e-12, "lượt ngược sai: hãy kiểm tra lại"

Lượt ngược khớp với sai phân hữu hạn ở mọi mảng, sai số tương đối dưới $10^{-6}$. Gradient của $b_K$ bằng 0 ở
mức làm tròn số, và đó không phải lỗi. Cộng cùng một vectơ $b_K$ vào mọi khóa làm mọi điểm trong một hàng tăng cùng một
lượng $q_t \cdot b_K$, mà softmax không đổi khi mọi đầu vào tăng như nhau. Vì vậy $b_K$ không ảnh hưởng gì đến đầu ra
(nhiều mô hình sau này bỏ hẳn độ lệch của Q, K, V). Mục 13 cho thấy nó vẫn “trôi” khi huấn luyện bằng Adam.

## 7 · Huấn luyện GPT nhỏ trên Truyện Kiều

Mỗi bước lấy ngẫu nhiên 16 khung 65 ký tự liên tiếp trong tập huấn luyện. Khung $x$ là 64 ký tự đầu, đáp án $y$ là
64 ký tự sau đó, lệch một vị trí. Vì có mặt nạ, **một lượt xuôi cho ra 64 bài toán “đoán ký tự kế tiếp”**, mỗi bài
chỉ thấy các ký tự đứng trước nó, và cả 64 cùng góp vào mất mát. Đây là lý do GPT huấn luyện song song được, còn RNN ở
Bài 15 phải đi lần lượt từng bước.

Bộ tối ưu là AdamW của Bài 07, với các hệ số như GPT-3 và Llama: $\beta_1 = 0{,}9$, $\beta_2 = 0{,}95$,
weight decay $\lambda = 0{,}1$ chỉ trên các ma trận $W$, không trên embedding, độ lệch và LayerNorm. Tốc độ học tăng
tuyến tính trong $5\%$ số bước đầu (warmup), rồi giảm theo cosin xuống còn $0{,}1\eta$. Tốc độ học $\eta = 0{,}04$ được
chọn trong $\{0{,}02;\ 0{,}03;\ 0{,}04;\ 0{,}06\}$ theo mất mát kiểm định, bằng chính hàm `train` dưới đây (mục 13 có
sẵn vòng quét để bạn tự chạy lại). Vì $\eta$ được chọn trên tập kiểm định và notebook không có tập kiểm tra riêng, con số ở cuối
hơi lạc quan.

In [ ]:
def val_bits(p, cfg):
    """Bit/ký tự trên 8.000 ký tự kiểm định, chia thành các khung 64 ký tự liền nhau."""
    seq = np.concatenate([train_ids[-1:], val_ids])  # ký tự đầu tiên của tập kiểm định cũng được đoán
    n = (len(seq) - 1) // cfg["ctx"]
    X = seq[: n * cfg["ctx"]].reshape(n, -1)
    Y = seq[1 : n * cfg["ctx"] + 1].reshape(n, -1)
    return cross_entropy(forward(p, X, cfg)[0], Y)[0] / math.log(2)


def lr_at(it, steps, eta):
    """Warmup tuyến tính trong 5% số bước đầu, rồi cosin từ η xuống 0,1·η."""
    warm = steps // 20
    if it <= warm:
        return eta * it / warm
    return eta * (0.1 + 0.45 * (1 + math.cos(math.pi * (it - warm) / (steps - warm))))


def adamw_step(p, g, state, lr, it, b1=0.9, b2=0.95, wd=0.1):
    """AdamW (Bài 07): weight decay tách riêng, chỉ trên các ma trận W."""
    for k_ in p:
        m, s = state.setdefault(k_, (np.zeros_like(p[k_]), np.zeros_like(p[k_])))
        m[:] = b1 * m + (1 - b1) * g[k_]
        s[:] = b2 * s + (1 - b2) * g[k_] * g[k_]
        if k_.split(".")[-1].startswith("W"):
            p[k_] *= 1 - lr * wd
        p[k_] -= lr * (m / (1 - b1**it)) / (np.sqrt(s / (1 - b2**it)) + 1e-8)


def train(cfg, steps=650, eta=0.04, B=16, seed=0, eval_every=65):
    p, state, rng = init_gpt(cfg, seed), {}, np.random.default_rng(seed)
    n = cfg["ctx"]
    log = {"step": [], "loss": [], "val_step": [0], "val": [val_bits(p, cfg)]}
    for it in range(1, steps + 1):
        start = rng.integers(0, len(train_ids) - n - 1, B)
        W = train_ids[start[:, None] + np.arange(n + 1)]  # 16 khung, mỗi khung 65 ký tự
        z, cache = forward(p, W[:, :-1], cfg)
        loss, dz = cross_entropy(z, W[:, 1:])  # 16 × 64 bài toán đoán ký tự kế tiếp cùng lúc
        adamw_step(p, backward(p, cache, dz, cfg), state, lr_at(it, steps, eta), it)
        log["step"].append(it)
        log["loss"].append(loss / math.log(2))
        if it % eval_every == 0 or it == steps:
            log["val_step"].append(it)
            log["val"].append(val_bits(p, cfg))
    return p, log


t0 = time.perf_counter()
params, log = train(CFG)
print(f"650 bước × 16 khung × 64 ký tự = {vn(650 * 16 * 64, 0)} lần đoán ({vn(time.perf_counter() - t0, 1)} giây)")
gpt_bits = log["val"][-1]
print(f"kiểm định: {vn(gpt_bits, 3)} bit/ký tự · bigram {vn(bigram_bits, 3)} · đoán đều {vn(uniform_bits, 3)}")
# dung sai rộng: float32 trên CPU khác (Colab là x86) làm lệch dần quỹ đạo huấn luyện
assert abs(gpt_bits - 2.557) < 0.05, "khác với lần chạy tham chiếu: hãy kiểm tra lại"

Hình dưới vẽ mất mát của từng lô huấn luyện (xám nhạt), trung bình trượt 25 bước của nó (xanh, nét liền) và mất mát trên
tập kiểm định sau mỗi 65 bước (cam, ô vuông). Hai đường gạch ngang là hai mốc của mục 2: đoán đều và bigram.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
loss = np.array(log["loss"])
smooth = np.convolve(loss, np.ones(25) / 25, mode="valid")
ax.plot(log["step"], loss, color=GREY, lw=0.6, alpha=0.5)
ax.plot(log["step"][24:], smooth, color=BLUE, lw=2, label="huấn luyện (trung bình 25 bước)")
ax.plot(log["val_step"], log["val"], color=ORANGE, marker="s", ls="--", lw=1.5, label="kiểm định")
for y, name in ((uniform_bits, "đoán đều"), (bigram_bits, "bigram")):
    ax.axhline(y, color="black", lw=0.8, ls=":")
    ax.text(650, y + 0.08, name, ha="right", fontsize=9)
ax.set(xlabel="bước", ylabel="bit mỗi ký tự", ylim=(2, 7.4), xlim=(0, 650))
ax.set_title("GPT nhỏ (72.516 tham số) học Truyện Kiều trong 650 bước", fontsize=11)
ax.legend(frameon=False, loc="upper right", bbox_to_anchor=(1, 0.9))
vn_axes(fig)
plt.show()
first_below = log["val_step"][next(i for i, b in enumerate(log["val"]) if b < bigram_bits)]
print(f"kiểm định xuống dưới bigram từ bước {first_below}; cuối cùng {vn(gpt_bits, 3)} bit/ký tự")
LSTM_15 = {"params": 150660, "steps": 4000, "bits": 2.2637}  # demo Bài 15 (rnn-kieu.json): LSTM H = 128, seed 0
print(
    f"để so, LSTM của Bài 15: {vn(LSTM_15['params'], 0)} tham số, tốt nhất trong {vn(LSTM_15['steps'], 0)} bước,"
    f" {vn(LSTM_15['bits'], 2)} bit/ký tự"
)

Mất mát bắt đầu ở 7,04 bit (đoán đều), vượt mốc bigram sau khoảng 200 bước, và dừng ở 2,56 bit sau 650 bước. Mất mát
huấn luyện và kiểm định đi sát nhau: mô hình 72.516 tham số chưa kịp học thuộc 96.204 ký tự. Để so: LSTM của Bài 15
(số của demo Bài 15, in ở trên) đạt 2,26 bit trên cùng 254 câu kiểm định, đo theo cách hơi khác: LSTM đọc liền một mạch nên ký tự nào cũng có đủ ngữ cảnh, còn ở đây ký tự đầu mỗi khung 64 ký tự
gần như không có ngữ cảnh. GPT nhỏ chỉ được tập 650 bước để cả notebook chạy dưới một phút; mục 13 gợi ý vài cách làm
nó tốt hơn.

## 8 · Mặt nạ che tương lai thật sự

Ta làm lại phép thử mà demo làm với GPT-2: chạy một câu thơ qua mô hình, rồi **đổi một ký tự ở giữa** và chạy lại.

### 🤔 Dự đoán trước
Đổi ký tự thứ 20 (chỉ số 19) của câu. Dự đoán ở 19 vị trí đứng trước nó thay đổi bao nhiêu? Còn ở các vị trí sau?

In [ ]:
line = "Trăm năm trong cõi người ta,\nChữ tài chữ mệnh khéo là ghét nhau."  # 64 ký tự: vừa một khung
x = np.array([stoi[c] for c in line])
x2 = x.copy()
x2[19] = stoi["x"]  # "người" → "xgười"
print(repr(decode(x[:24])), "→", repr(decode(x2[:24])))
for causal in (True, False):
    za = forward(params, x[None], CFG, causal)[0][0]
    zb = forward(params, x2[None], CFG, causal)[0][0]
    delta = np.abs(zb - za).max(-1)  # thay đổi lớn nhất của logit ở mỗi vị trí
    name = "có mặt nạ  " if causal else "bỏ mặt nạ "
    print(
        f"{name}: vị trí 0–18 đổi tối đa {sci(delta[:19].max())} · vị trí 19–{len(x) - 1} đổi tối đa {vn(delta[19:].max())}"
    )
    if causal:
        assert delta[:19].max() == 0.0 and delta[19:].max() > 0.1

Có mặt nạ, dự đoán ở 19 vị trí đứng trước chỗ sửa **không đổi một bit nào** (sai khác đúng bằng 0, không chỉ là nhỏ):
chúng không bao giờ đọc ký tự thứ 20. Bỏ mặt nạ trên cùng trọng số thì sửa một ký tự phía sau làm đổi cả các dự đoán
phía trước. Mặt nạ là thứ cho phép huấn luyện cả 64 vị trí trong một lượt mà không vị trí nào được nhìn trước đáp án
của nó; và vì vậy nó được dùng *cả khi huấn luyện*, không chỉ khi sinh văn bản. Demo làm đúng phép thử này với GPT-2:
đổi “ largest” thành “ oldest” và 19 phân phối đứng trước giữ nguyên (mục 12).

## 9 · Sinh văn bản: nhiệt độ, top-k và bộ nhớ đệm KV

Huấn luyện thì song song, nhưng sinh văn bản vẫn là **từng token một**: đoán phân phối của token kế tiếp, rút một
token, nối vào chuỗi, lặp lại. Ở bước rút, logit $z$ được chia cho **nhiệt độ** $T$ trước softmax:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

$T < 1$ làm phân phối nhọn hơn, $T > 1$ phẳng hơn, $T \to 0$ là luôn chọn token có logit lớn nhất (tham lam). **Top-k**
chỉ giữ $k$ token có logit lớn nhất rồi chuẩn hóa lại. Chia cho một số dương không đổi thứ tự các logit, nên $T$ không
bao giờ đổi token đứng đầu; nó chỉ đổi khoảng cách giữa các token.

**Bộ nhớ đệm KV.** Sinh token thứ $t+1$ bằng cách chạy lại cả chuỗi là phí: khóa và giá trị của các vị trí cũ không đổi
(mặt nạ bảo đảm vị trí cũ không bao giờ nhìn token mới). Vậy ta tính cả câu lệnh một lần (*prefill*), cất $K$, $V$ của
mọi khối, rồi mỗi bước chỉ tính cho **một** vị trí mới: truy vấn của nó chấm với mọi khóa đã cất, không cần mặt nạ.

In [ ]:
def prefill(p, prompt, cfg):
    """Một lượt có mặt nạ trên cả câu lệnh: logit ở vị trí cuối, cùng K, V của mọi khối (đầu, vị trí, d_h)."""
    z, cache = forward(p, np.array(prompt)[None], cfg)
    return z[0, -1], [c["k"][0] for c in cache["layers"]], [c["v"][0] for c in cache["layers"]]


def decode_step(p, token, pos, K, Vc, cfg):
    """Một token mới ở vị trí pos: tính q, k, v của riêng nó, nối k, v vào bộ nhớ đệm, chú ý tới mọi vị trí đã có."""
    d, nh = cfg["d"], cfg["heads"]
    h = p["wte"][token] + p["wpe"][pos]  # (d,)
    for l in range(cfg["L"]):
        a, _ = layernorm(h, p[f"{l}.g1"], p[f"{l}.b1"])
        q, k, v = (a @ p[f"{l}.Wqkv"] + p[f"{l}.bqkv"]).reshape(3, nh, d // nh)
        K[l] = np.concatenate([K[l], k[:, None]], axis=1)  # cất khóa, giá trị của vị trí mới
        Vc[l] = np.concatenate([Vc[l], v[:, None]], axis=1)
        o, _ = attention(q[:, None], K[l], Vc[l], causal=False)  # một truy vấn, mọi khóa đã có: không cần mặt nạ
        h = h + o.reshape(d) @ p[f"{l}.Wo"] + p[f"{l}.bo"]
        m, _ = layernorm(h, p[f"{l}.g2"], p[f"{l}.b2"])
        h = h + gelu(m @ p[f"{l}.W1"] + p[f"{l}.c1"])[0] @ p[f"{l}.W2"] + p[f"{l}.c2"]
    hf, _ = layernorm(h, p["gf"], p["bf"])
    return hf @ p["Wh"] + p["bh"]

### 🤔 Dự đoán trước
Sinh tham lam 54 ký tự sau câu lệnh 10 ký tự “Trăm năm t”, một lần bằng bộ nhớ đệm KV và một lần bằng cách chạy lại cả
chuỗi. Hai cách có cho cùng một đoạn thơ không? Logit lệch nhau bao nhiêu?

In [ ]:
prompt = [stoi[c] for c in "Trăm năm t"]
z, K, Vc = prefill(params, prompt, CFG)
seq, cached = list(prompt), [z]
t0 = time.perf_counter()
while len(seq) < CFG["ctx"]:
    seq.append(int(np.argmax(cached[-1])))  # tham lam
    if len(seq) < CFG["ctx"]:
        cached.append(decode_step(params, seq[-1], len(seq) - 1, K, Vc, CFG))
t_cache = time.perf_counter() - t0
t0 = time.perf_counter()
full = [forward(params, np.array(seq[:t])[None], CFG)[0][0, -1] for t in range(len(prompt), len(seq))]  # chạy lại
t_full = time.perf_counter() - t0
gap = np.abs(np.array(cached) - np.array(full)).max()
greedy_full = [int(np.argmax(f)) for f in full]
print(repr(decode(seq)))
print(f"cùng {len(seq) - len(prompt)} ký tự: {greedy_full == seq[len(prompt) :]} · logit lệch nhau tối đa {sci(gap)}")
print(
    f"(thời gian trên máy này: có bộ nhớ đệm {vn(t_cache * 1000, 0)} ms, chạy lại cả chuỗi {vn(t_full * 1000, 0)} ms)"
)
assert greedy_full == seq[len(prompt) :] and gap < 1e-4

Cùng 54 ký tự, logit lệch nhau chỉ ở mức làm tròn của float32: bộ nhớ đệm KV **không phải phép xấp xỉ**, nó tính đúng
những con số của lượt đầy đủ, chỉ là không tính lại phần đã có. Cái giá là bộ nhớ: mỗi token đã đọc giữ một khóa và một
giá trị ở mọi khối (mục 10 đếm cho GPT-2).

Đoạn thơ tham lam cũng cho thấy một tật quen thuộc: luôn chọn ký tự chắc nhất làm mô hình lặp lại chính nó (ở lần chạy
tham chiếu: “thôi thôi … thôi”, “thế thế”). Vì vậy người ta *rút ngẫu nhiên*. Ô dưới sinh 1.000 ký tự cho mỗi cách rút, sau 4 câu
cuối của tập huấn luyện, rồi đo hai thứ: tỉ lệ câu có đúng 6 hoặc 8 tiếng (thể lục bát), và tỉ lệ tiếng là tiếng thật,
tức có xuất hiện trong tập huấn luyện. Để rút ký tự kế tiếp, mô hình chỉ đọc 64 ký tự cuối (cả khung ngữ cảnh của nó).

In [ ]:
SYLLABLES = {w for ln in LINES[:3000] for w in re.sub(r"[^\w\s]", " ", ln.lower()).split()}


def generate(p, prompt, n, cfg, T=1.0, k=0, seed=0):
    """n token sau prompt; T = 0 là tham lam, k > 0 là top-k. Mỗi bước chỉ đọc cfg["ctx"] token cuối."""
    rng = np.random.default_rng(seed)
    seq = list(prompt)
    for _ in range(n):
        z = forward(p, np.array(seq[-cfg["ctx"] :])[None], cfg)[0][0, -1].astype(np.float64)
        if T == 0:
            seq.append(int(np.argmax(z)))
            continue
        keep = np.argsort(-z, kind="stable")[:k] if k else np.arange(len(z))  # top-k: chỉ giữ k logit lớn nhất
        seq.append(int(keep[rng.choice(len(keep), p=softmax(z[keep] / T))]))
    return seq[len(prompt) :]


def poem_stats(s):
    """Tỉ lệ câu 6 hoặc 8 tiếng và tỉ lệ tiếng có trong tập huấn luyện (bỏ câu cuối, có thể bị cắt dở)."""
    words = [re.sub(r"[^\w\s]", " ", ln.lower()).split() for ln in s.split("\n")[:-1]]
    flat_words = [w for ws in words for w in ws]
    return len(words), np.mean([len(ws) in (6, 8) for ws in words]), np.mean([w in SYLLABLES for w in flat_words])


cue = [stoi[c] for c in "\n".join(LINES[2996:3000]) + "\n"]
N_GEN = 1000
print(f"mỗi cách rút sinh {vn(N_GEN, 0)} ký tự")
print(f"thơ thật (254 câu kiểm định): {pct(poem_stats(chr(10).join(LINES[3000:]) + chr(10))[1], 0)} câu 6/8 tiếng")
rng = np.random.default_rng(0)
bi = [stoi["\n"]]
for _ in range(N_GEN):  # bigram, T = 1: chỉ nhìn ký tự liền trước
    bi.append(int(rng.choice(V, p=P_bigram[bi[-1]])))
for name, s in [("bigram, T = 1", decode(bi[1:]))] + [
    (f"GPT nhỏ, T = {vn(T, 1)}" + (f", top-k {k}" if k else ""), decode(generate(params, cue, N_GEN, CFG, T, k)))
    for T, k in ((0.5, 0), (1.0, 0), (1.0, 5))
]:
    n_lines, share68, real = poem_stats(s)
    print(f"{name:<22} {n_lines:>3} câu · {pct(share68, 0):>4} câu 6/8 tiếng · {pct(real, 0):>4} tiếng thật")
    print("   ", " / ".join(s.split("\n")[:3]))

Mỗi dòng in số liệu của 1.000 ký tự, kèm 3 câu đầu. Mô hình bigram ra những chữ không có thật, và câu dài ngắn tùy ý.
GPT nhỏ đã học được nhiều hơn hẳn: chữ hoa đầu câu, câu ngắt bằng dấu phẩy hay dấu chấm, độ dài câu gần với thể lục
bát, và phần lớn tiếng là tiếng thật, dù câu thơ chưa có nghĩa. Nhiệt độ thấp ($T = 0{,}5$) cho nhiều tiếng thật nhất;
$T = 1$ rút cả những ký tự ít khả năng, nên nhiều tiếng sai chính tả hơn. Top-k 5 giữ $T = 1$ nhưng cắt đuôi phân phối,
và tỉ lệ tiếng thật tăng lại. Mỗi con số chỉ đến từ một lần rút với seed 0, nên chênh lệch vài phần trăm chưa nói lên
điều gì; điều đáng nhìn là chiều của chúng.

## 10 · Cái giá của GPT-2: tham số, phép tính, bộ nhớ

GPT nhỏ có 72.516 tham số. GPT-2 small là cùng kiến trúc với cấu hình đã công bố: 12 khối, $d = 768$, 12 đầu × 64 chiều,
MLP $768 \to 3.072 \to 768$, từ vựng 50.257 token BPE (Bài 14), khung 1.024 token, và lớp đầu ra dùng lại $E^\top$
(không có $W_h$ riêng). Ô dưới đếm tham số từng phần, so với số đo trên trọng số thật của demo.

In [ ]:
GPT2 = {"L": 12, "d": 768, "heads": 12, "ff": 3072, "V": 50257, "ctx": 1024}  # hparams.json của OpenAI


def gpt2_params(c):
    d, ff, L = c["d"], c["ff"], c["L"]
    return {
        "E: embedding token": c["V"] * d,
        "P: embedding vị trí": c["ctx"] * d,
        "chú ý: W_QKV, b_QKV": L * (d * 3 * d + 3 * d),
        "chú ý: W_O, b_O": L * (d * d + d),
        "MLP: W_1, c_1": L * (d * ff + ff),
        "MLP: W_2, c_2": L * (ff * d + d),
        "LayerNorm: g, b": (2 * L + 1) * 2 * d,
        "đầu ra: dùng lại Eᵀ": 0,
    }


parts = gpt2_params(GPT2)
total = sum(parts.values())
for name, n_ in parts.items():
    print(f"{name:<22} {vn(n_, 0):>11}  {pct(n_ / total, 1):>6}")
print(f"{'tổng':<22} {vn(total, 0):>11}")
attn, mlp = parts["chú ý: W_QKV, b_QKV"] + parts["chú ý: W_O, b_O"], parts["MLP: W_1, c_1"] + parts["MLP: W_2, c_2"]
block_ln = GPT2["L"] * 2 * 2 * GPT2["d"]  # hai LayerNorm mỗi khối
print(
    f"trong các khối, MLP chiếm {pct(mlp / (attn + mlp + block_ln), 1)} tham số, chú ý {pct(attn / (attn + mlp + block_ln), 1)}"
)
real = inside["params"]  # đếm trên các mảng trọng số thật (demo)
print(f"đếm trên trọng số thật: {vn(real['total'], 0)} · chú ý {vn(real['attn'], 0)} · MLP {vn(real['mlp'], 0)}")
assert total == 124439808 and (attn, mlp, parts["E: embedding token"]) == (real["attn"], real["mlp"], real["wte"])

Đúng 124.439.808, bằng số đếm trên các mảng trọng số thật. (Bài báo GPT-2 ghi 117 triệu; README của OpenAI sau đó
đính chính rằng các con số ban đầu bị đếm sai, và bản nhỏ nay được gọi là 124M.) Bảng embedding chiếm gần một phần ba
mô hình, vì từ vựng lớn; trong các khối, MLP chiếm khoảng 2/3 tham số, đúng như trang bài học nói.

**Phép tính.** Theo quy ước của demo, một phép nhân ma trận $d_\text{vào} \times d_\text{ra}$ tốn $d_\text{vào}\,d_\text{ra}$
phép nhân–cộng (MAC) cho mỗi vị trí; độ lệch, LayerNorm và softmax không tính. Mỗi vị trí đi qua các ma trận của 12 khối
(một lượng cố định), chú ý tới $t$ vị trí ($q \cdot k$ rồi cộng các $v$: $2 \cdot 12 \cdot 768 \cdot t$ MAC), và lớp đầu
ra chỉ cần cho token được sinh. Ô dưới viết công thức sinh $N$ token sau câu lệnh $P$ token, có và không có bộ nhớ đệm, và
kiểm nó bằng cách đếm lại từng vị trí một.

In [ ]:
def dense_mac(c):  # Q, K, V, W_O và MLP của mọi khối, cho một vị trí
    return c["L"] * (c["d"] * 3 * c["d"] + c["d"] * c["d"] + 2 * c["d"] * c["ff"])


def attn_mac(c, t):  # một vị trí chú ý tới t vị trí (kể cả chính nó)
    return 2 * c["L"] * c["d"] * t


def out_mac(c):
    return c["d"] * c["V"]


def gen_mac(c, P, N, cache):
    """Sinh N token sau câu lệnh P token. Có bộ nhớ đệm: mỗi vị trí đi qua mô hình đúng một lần (token cuối cùng được
    sinh ra thì không cần đưa vào nữa). Không có: bước thứ i chạy lại cả P + i vị trí."""
    tri = attn_mac(c, 1)  # Σ_{t=1}^{n} attn_mac(t) = tri · n(n + 1)/2
    if cache:
        n = P + N - 1
        return n * dense_mac(c) + tri * n * (n + 1) // 2 + N * out_mac(c)
    return sum((P + i) * dense_mac(c) + tri * (P + i) * (P + i + 1) // 2 for i in range(N)) + N * out_mac(c)


def recount(c, P, N, cache):  # đếm lại từng vị trí đã xử lý
    passes = [range(1, P + N)] if cache else [range(1, P + i + 1) for i in range(N)]
    return sum(dense_mac(c) + attn_mac(c, t) for r in passes for t in r) + N * out_mac(c)


print(f"mỗi vị trí: {vn(dense_mac(GPT2), 0)} MAC trong các khối + {vn(out_mac(GPT2), 0)} MAC ở lớp đầu ra")
print(
    f"   = {vn((dense_mac(GPT2) + out_mac(GPT2)) / 1e6, 1)} triệu MAC cho một token mới (câu còn ngắn);"
    f" chú ý ở vị trí 1.024 thêm {vn(attn_mac(GPT2, 1024) / 1e6, 1)} triệu"
)
for P, N in ((6, 30), (6, 1018)):
    w, wo = gen_mac(GPT2, P, N, True), gen_mac(GPT2, P, N, False)
    assert w == recount(GPT2, P, N, True) and wo == recount(GPT2, P, N, False)
    print(
        f"câu lệnh {P} + sinh {vn(N, 0)} token: có bộ nhớ đệm {vn(w / 1e9, 1)} tỉ MAC, chạy lại {vn(wo / 1e9, 1)} tỉ"
        f" (gấp {vn(wo / w, 0)} lần)"
    )
kv_bytes = 2 * GPT2["L"] * GPT2["d"] * 2  # một khóa và một giá trị, 12 khối × 768 số, fp16 = 2 byte
print(
    f"bộ nhớ đệm KV: {vn(kv_bytes / 1024, 0)} KiB mỗi token (fp16), {vn(kv_bytes * 1024 / 2**20, 0)} MiB cho 1.024 token"
)
print(f"GPT nhỏ của ta: {2 * CFG['L'] * CFG['d'] * 4} byte mỗi token (float32)")
assert (dense_mac(GPT2), out_mac(GPT2)) == (84934656, 38597376) and kv_bytes == 36864
assert (gen_mac(GPT2, 6, 30, True), gen_mac(GPT2, 6, 30, False)) == (4142246400, 53535306240)

Đây đúng là các con số demo hiển thị cho câu lệnh 6 token của reel. Khi câu còn ngắn, mỗi token mới tốn khoảng
123,5 triệu MAC, xấp xỉ **một MAC cho mỗi tham số** (Kaplan và cộng sự, 2020, viết là $\approx 2N$ FLOP mỗi token).
Phần chú ý tăng theo độ dài: ở vị trí 1.024 nó thêm $2 \cdot 12 \cdot 768 \cdot 1.024$ MAC, khoảng 18,9 triệu.
Không có bộ nhớ đệm, tổng công tăng theo bình phương số token sinh ra: hình dưới (thang log cả hai trục) vẽ tổng MAC để
sinh $N$ token sau câu lệnh 6 token, có bộ nhớ đệm (xanh, chấm tròn) và chạy lại cả chuỗi (cam, ô vuông, nét gạch).

In [ ]:
Ns = np.unique(np.geomspace(1, 1018, 30).astype(int))
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(Ns, [gen_mac(GPT2, 6, N, True) for N in Ns], color=BLUE, marker="o", ms=4, label="có bộ nhớ đệm KV")
ax.plot(
    Ns, [gen_mac(GPT2, 6, N, False) for N in Ns], color=ORANGE, marker="s", ms=4, ls="--", label="tính lại cả chuỗi"
)
ax.set(xscale="log", yscale="log", xlabel="số token sinh ra N", ylabel="tổng số MAC")
ax.set_title("GPT-2 small, câu lệnh 6 token: tổng phép nhân–cộng để sinh N token", fontsize=11)
ax.legend(frameon=False)
ax.grid(alpha=0.3, which="major")
vn_axes(fig)
plt.show()

Hai đường bắt đầu như nhau rồi tách xa: với $N$ = 1.018 token, chạy lại cả chuỗi tốn gấp 352 lần. Bộ nhớ đệm KV đổi
phép tính lấy bộ nhớ. GPT-2 cần 36 KiB mỗi token (fp16) cho mỗi chuỗi đang sinh, nhân với số chuỗi phục vụ cùng lúc.
Với các mô hình lớn hơn, chính bộ nhớ này giới hạn số người dùng một GPU phục vụ được (mục 14).

## 11 · Đầu quy nạp: vì sao cần ít nhất hai khối

Sau huấn luyện, nhiều đầu của GPT-2 làm những việc dễ gọi tên (mục 12). Việc nổi tiếng nhất là **chép**: gặp lại
token [A], nhìn xem lần trước sau [A] là gì ([B]), rồi đoán [B]. Olsson và cộng sự (2022) gọi đầu làm việc này là
**đầu quy nạp** (induction head). Nó cần hai đầu ở hai khối nối tiếp nhau:
1. khối 0 có một **đầu nhìn token liền trước**, chép “token đứng trước tôi là [A]” vào vectơ của vị trí chứa [B];
2. ở khối 1, truy vấn của vị trí hiện tại (token [A]) tìm vị trí có ghi “đứng trước tôi là [A]”, tức chính vị trí [B],
   rồi chép [B] ra.

Ta kiểm điều đó trên một bài toán thuần túy: dãy 25 ký hiệu ngẫu nhiên (16 loại), trong đó một đoạn dài 3–8 ký hiệu
được lặp lại ngay sau chính nó. Mất mát chỉ tính trên những ký hiệu của lần lặp mà đoạn trước cho biết trước (từ ký hiệu
thứ hai của lần lặp). Đoán bừa tốn $\log_2 16 = 4$ bit. Mô hình nhỏ: $d = 32$, 2 đầu, khung 24 vị trí; một bản có 2 khối,
một bản có 1 khối, cùng hàm `forward`, `backward` và AdamW ở trên.

In [ ]:
def copy_batch(rng, B, n_pos=24, n_sym=16):
    """Dãy ngẫu nhiên có một đoạn dài 3–8 lặp lại ngay sau nó. w = 1 ở những vị trí mà đoạn trước cho biết đáp án."""
    X = rng.integers(0, n_sym, (B, n_pos + 1))
    w = np.zeros((B, n_pos))
    seg = []
    for b in range(B):
        n_ = rng.integers(3, 9)
        s0 = rng.integers(0, n_pos + 1 - 2 * n_)
        X[b, s0 + n_ : s0 + 2 * n_] = X[b, s0 : s0 + n_]  # lần lặp
        w[b, s0 + n_ : s0 + 2 * n_ - 1] = 1  # đoán ký hiệu thứ 2…n của lần lặp
        seg.append((s0, n_))
    return X, w, seg


def train_copy(L, steps=600, eta=0.01, seed=0):
    cfg = {"V": 16, "ctx": 24, "d": 32, "L": L, "heads": 2}
    p, state, rng = init_gpt(cfg, seed), {}, np.random.default_rng(seed)
    Xe, we, _ = copy_batch(np.random.default_rng(99), 256)  # lô kiểm tra cố định
    curve = []
    for it in range(1, steps + 1):
        X, w, _ = copy_batch(rng, 32)
        z, cache = forward(p, X[:, :-1], cfg)
        _, dz = cross_entropy(z, X[:, 1:], w)
        adamw_step(p, backward(p, cache, dz, cfg), state, lr_at(it, steps, eta), it)
        if it % 50 == 0:
            curve.append(cross_entropy(forward(p, Xe[:, :-1], cfg)[0], Xe[:, 1:], we)[0] / math.log(2))
    return p, cfg, curve

### 🤔 Dự đoán trước
Cả hai mô hình được huấn luyện 600 bước như nhau. Mô hình **một khối**, với 2 đầu chú ý, có học được cách chép không?

In [ ]:
t0 = time.perf_counter()
copy_runs = {L: train_copy(L) for L in (1, 2)}
for L, (_, _, curve) in copy_runs.items():
    print(f"{L} khối: lần lặp tốn {vn(curve[0], 2)} → {vn(curve[-1], 2)} bit/ký hiệu (đoán bừa: 4)")
print(f"({vn(time.perf_counter() - t0, 1)} giây)")
assert copy_runs[2][2][-1] < 1.5 and copy_runs[1][2][-1] > 2.0, "khác với lần chạy tham chiếu"

fig, ax = plt.subplots(figsize=(7, 3.4))
steps_ = np.arange(50, 601, 50)
ax.plot(steps_, copy_runs[1][2], color=ORANGE, marker="s", ms=4, ls="--", label="1 khối")
ax.plot(steps_, copy_runs[2][2], color=BLUE, marker="o", ms=4, label="2 khối")
ax.axhline(4, color="black", lw=0.8, ls=":")
ax.text(600, 4.05, "đoán bừa", ha="right", va="bottom", fontsize=9)
ax.set(xlabel="bước", ylabel="bit mỗi ký hiệu của lần lặp", ylim=(0, 4.4), xlim=(0, 600))
ax.set_title("Chép đoạn lặp: mô hình 1 khối và 2 khối", fontsize=11)
ax.legend(frameon=False, loc="lower left")
vn_axes(fig)
plt.show()

Mô hình hai khối học được cách chép: lần lặp còn khoảng 1 bit mỗi ký hiệu hoặc ít hơn, so với 4 bit của đoán bừa. Mô hình một khối
dừng ở khoảng 2,5 bit, tốt hơn đoán bừa nhưng không chép được. Một khối chỉ biết “ký hiệu nào *đã xuất hiện*” (nó có thể nhìn lại mọi ký hiệu
trước đó và tăng xác suất của chúng), nhưng không biết ký hiệu nào *đứng sau* lần xuất hiện trước của token hiện tại:
thông tin “đứng trước tôi là gì” chưa có trong vectơ của vị trí nào, vì phải có một lớp chú ý đặt nó vào đó trước.

Ô dưới mở mô hình hai khối ra. Với mỗi đầu, nó đo hai điểm trên lô kiểm tra: **điểm token liền trước** (trung bình
$\alpha_{t,t-1}$) và **điểm quy nạp** (trung bình sự chú ý mà một vị trí của lần lặp đặt vào đúng vị trí chứa đáp án
trong đoạn gốc), rồi vẽ ma trận chú ý của một dãy: bên trái là đầu nhìn token liền trước ở khối 0, bên phải là đầu quy
nạp ở khối 1. Màu đậm là trọng số lớn; vòng tròn trắng đánh dấu các ô mà mỗi kiểu đầu “nên” nhìn vào.

In [ ]:
p_copy, cfg_copy, _ = copy_runs[2]
Xe, _, seg = copy_batch(np.random.default_rng(99), 256)
_, cache = forward(p_copy, Xe[:, :-1], cfg_copy)
prev_score, ind_score = [], []
for c in cache["layers"]:
    A_ = c["A"]  # (lô, đầu, 24, 24)
    prev_score.append(A_[:, :, np.arange(1, 24), np.arange(0, 23)].mean((0, 2)))
    hits = [A_[b, :, t, t - n_ + 1] for b, (s0, n_) in enumerate(seg) for t in range(s0 + n_, s0 + 2 * n_ - 1)]
    ind_score.append(np.mean(hits, axis=0))
for l in range(2):
    print(
        f"khối {l}: điểm token liền trước {' / '.join(vn(x, 2) for x in prev_score[l])} (đầu 0 / đầu 1)"
        f" · điểm quy nạp {' / '.join(vn(x, 2) for x in ind_score[l])}"
    )
h_prev, h_ind = int(np.argmax(prev_score[0])), int(np.argmax(ind_score[1]))
assert prev_score[0][h_prev] > 0.25 and ind_score[1][h_ind] > 0.4 > ind_score[0].max()

b = next(i for i, (s0, n_) in enumerate(seg) if n_ >= 6)  # một dãy có đoạn lặp dài
s0, n_ = seg[b]
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.4))
for ax, l, hd, title in (
    (axes[0], 0, h_prev, "khối 0, đầu {}: nhìn token liền trước"),
    (axes[1], 1, h_ind, "khối 1, đầu {}: đầu quy nạp"),
):
    ax.imshow(cache["layers"][l]["A"][b, hd], cmap=SEQ, vmin=0, vmax=1)
    want = [(t, t - 1) for t in range(1, 24)] if l == 0 else [(t, t - n_ + 1) for t in range(s0 + n_, s0 + 2 * n_ - 1)]
    ax.scatter([j for _, j in want], [t for t, _ in want], s=70, facecolors="none", edgecolors="white", lw=1.2)
    ax.set(xlabel="vị trí được nhìn j", ylabel="vị trí truy vấn t", title=title.format(hd))
    ax.set_xticks(range(0, 24, 4))
    ax.set_yticks(range(0, 24, 4))
fig.suptitle(
    f"Một dãy có đoạn {n_} ký hiệu ở vị trí {s0}–{s0 + n_ - 1}, lặp lại ở {s0 + n_}–{s0 + 2 * n_ - 1}", fontsize=11
)
plt.tight_layout()
vn_axes(fig)
plt.show()

Đúng như sơ đồ. Ở khối 0, cả hai đầu đều nhìn token liền trước (khoảng một phần ba sự chú ý, gấp nhiều lần mức trung
bình), ô sáng nằm ngay dưới đường chéo. Ở khối 1, hai đầu đặt khoảng 0,6 tổng trọng số chú ý vào đúng vị trí chứa đáp
án trong đoạn gốc: các ô sáng chạy song song với đường chéo, cách nó đúng một độ dài đoạn lặp. Không ai lập trình hai việc
này; chúng tự xuất hiện vì giúp giảm mất mát. GPT-2 có cùng mạch đó, ở quy mô lớn hơn (mục 12).

## 12 · Bên trong GPT-2 thật

Từ đây ta đọc các số đo trên GPT-2 small mà demo dùng (`inside`, `samp`, nạp ở mục 2). Token của GPT-2 là token BPE
cấp byte (Bài 14). Token bắt đầu bằng dấu cách (“ Paris”) là một từ mới. Một chữ tiếng Việt có dấu thường bị cắt thành
nhiều mảnh byte UTF-8; notebook in các mảnh chưa thành chữ dưới dạng mã hex trong ngoặc vuông.

**Một lượt, mọi vị trí.** Như GPT nhỏ ở mục 7, một lượt có mặt nạ qua câu dưới cho ra một dự đoán ở mỗi vị trí. Ô dưới in,
ở mỗi vị trí, token GPT-2 đoán là có khả năng nhất, cùng xác suất và thứ hạng mà nó cho token thật đứng kế tiếp.

In [ ]:
TOK = {}
for tokens in (inside["tokens"], samp["tokens"]):
    TOK |= {int(i): s for i, s in tokens["text"].items()} | {int(i): f"[{h}]" for i, h in tokens["raw"].items()}

r_en, r_vi = inside["read"]["en"], inside["read"]["vi"]
toks = [TOK[i] for i in r_en["ids"]]
print(r_en["text"])
for t, pos in enumerate(r_en["pos"]):
    (best, p_best), hit = pos["top"][0], "  ← đúng" if pos["rank"] == 1 else ""
    print(
        f"{t:>2} sau “{toks[t]}”: đoán “{TOK[best]}” {pct(p_best, 1):>6} · token thật “{toks[t + 1]}” hạng"
        f" {vn(pos['rank'], 0)}, {pct(pos['p'], 2)}{hit}"
    )
hits = sum(pos["rank"] == 1 for pos in r_en["pos"])
print(f"đúng hạng 1 ở {hits}/{len(r_en['pos'])} vị trí · trung bình {vn(r_en['bitsPerToken'], 2)} bit/token")

Một lượt cho cả 27 dự đoán. Có những chỗ GPT-2 gần như chắc chắn (“ Asia” sau “ Southeast”, “ City” sau “ Ho Chi
Minh”), và cũng có chỗ nó không thể biết trước (token đầu tiên, hay “ is” sau “Vietnam”). Ở vị trí đầu, mô hình chỉ
có đúng một token để đọc.

Bây giờ là phép thử của mục 8, trên GPT-2: đổi “ largest” thành “ oldest”. Ô dưới in mức thay đổi lớn nhất của logit
ở mỗi vị trí, và so câu tiếng Anh với bản dịch tiếng Việt của nó.

In [ ]:
e = inside["read"]["en"]["edit"]
print(f"đổi “{TOK[e['from']]}” → “{TOK[e['to']]}” ở token {e['pos']}:")
print(f"  vị trí 0–{e['pos'] - 1}: Δ logit lớn nhất {vn(max(e['delta'][: e['pos']]), 3)}")
print(f"  vị trí {e['pos']}–{len(e['delta']) - 1}: {' · '.join(vn(x, 1) for x in e['delta'][e['pos'] :])}")
print(f"  sau “ oldest”, GPT-2 đoán: {', '.join(f'“{TOK[i]}” {pct(q, 1)}' for i, q in e['top'][0])}")
for name, r_ in (("tiếng Anh", r_en), ("tiếng Việt", r_vi)):
    print(
        f"{name:<10}: {r_['bytes']} byte → {len(r_['ids'])} token · {vn(r_['bitsPerToken'], 2)} bit/token ·"
        f" {vn(r_['bitsPerByte'], 2)} bit/byte"
    )
print("10 token đầu của câu tiếng Việt:", " | ".join(TOK[i] for i in r_vi["ids"][:10]))
for lang, fl in inside["read"]["flores"].items():
    print(
        f"FLORES-200 ({lang}, {fl['sentences']} câu): {vn(fl['bitsPerByte'], 2)} bit/byte, {vn(fl['bytes'] / fl['tokens'], 2)} byte/token"
    )

Có mặt nạ, 19 phân phối đứng trước “ oldest” giữ nguyên tuyệt đối (Δ = 0), đúng như GPT nhỏ ở mục 8. Từ vị trí bị sửa
trở đi, logit đổi tới cỡ 10.

Câu tiếng Việt cùng nghĩa dài gấp hơn 3 lần về số token, và tốn hơn 4 lần số bit cho mỗi byte. Bộ tách token BPE của
GPT-2 học từ văn bản tiếng Anh nên cắt nhiều chữ có dấu thành từng byte UTF-8 (“Việt” thành “Vi”, ba mảnh byte của “ệ”,
rồi “t”). Bài báo GPT-2 cũng nói rõ họ đã cố ý lọc bỏ các trang không phải tiếng Anh khỏi WebText. Trên 40 câu
FLORES-200 mỗi ngôn ngữ cũng vậy: mỗi token tiếng Việt chỉ chứa 1,55 byte (tiếng Anh 4,70), và mỗi byte tốn gần gấp
3 lần số bit.

### Logit lens: đọc dòng dư sau từng khối

Lớp đầu ra chỉ đọc vectơ cuối cùng của dòng dư. Nhưng có thể thử đọc *sớm*: lấy vectơ ở vị trí cuối sau khối $\ell$, đưa
qua LayerNorm cuối rồi nhân $E^\top$, như thể mô hình dừng ở đó (nostalgebraist, 2020). Ô dưới làm việc này cho ba câu
lệnh và in thứ hạng của đáp án ở mỗi độ sâu (0 là ngay sau embedding, 12 là đầu ra thật).

In [ ]:
def gpt_text(token_ids):
    return "".join(TOK[i] for i in token_ids)


for x in inside["lens"]:
    ranks = [dp["rank"] for dp in x["depth"]]
    first = next(i for i in range(len(ranks)) if all(r == 1 for r in ranks[i:]))
    print(f"“{gpt_text(x['ids'])}” → “{TOK[x['answer']]}”")
    print(f"   hạng theo độ sâu 0–12: {' · '.join(vn(r, 0) for r in ranks)}")
    tops = " → ".join(f"{L}: “{TOK[x['depth'][L]['top'][0][0]]}”" for L in (0, 6, 9, 10, 11, 12))
    print(f"   đứng đầu từ độ sâu {first}; token đứng đầu ở độ sâu {tops}")
    print(
        f"   xác suất ở đầu ra {pct(x['p'], 2)}; độ dài vectơ {vn(x['depth'][0]['norm'], 1)} → {vn(x['depth'][-1]['norm'], 1)}"
    )

fig, ax = plt.subplots(figsize=(7.6, 3.8))
for x, c, mk, ls in zip(inside["lens"], (BLUE, ORANGE, GREEN), "os^", ("-", "--", ":")):
    ax.plot(range(13), [dp["rank"] for dp in x["depth"]], color=c, marker=mk, ls=ls, label=f"“{TOK[x['answer']]}”")
ax.set(yscale="log", xlabel="số khối đã đi qua", ylabel="hạng của đáp án (1 = đứng đầu)", xlim=(-0.3, 12.3))
ax.invert_yaxis()
ax.set_xticks(range(13))
ax.set_title("Logit lens trên GPT-2 small: đáp án leo lên đầu ở các khối cuối", fontsize=11)
ax.legend(frameon=False, title="đáp án", loc="lower right")
vn_axes(fig)
plt.show()

Ba đường (xanh nét liền “ Paris”, cam nét gạch “ Mary”, xanh lá chấm “ visualize”; trục dọc theo thang log, hạng 1 ở
trên cùng) kể cùng một chuyện. Ngay sau embedding, đáp án đứng hạng hàng nghìn đến hàng chục nghìn. Nó leo dần qua các
khối giữa và chỉ đứng đầu sau 10–11 trong 12 khối. Với câu Tháp Eiffel, sau 10 khối vectơ còn đọc ra “ London”, sau
11 khối mới là “ Paris”. Độ dài vectơ ở vị trí cuối tăng dần qua các khối: mỗi lớp con *cộng thêm* vào dòng dư chứ không
thay thế nó.
Logit lens chỉ là một cách nhìn gần đúng: các khối giữa không được huấn luyện để đọc qua $E^\top$, và *tuned lens*
(Belrose và cộng sự, 2023) học riêng một phép chiếu cho mỗi khối để đọc chính xác hơn.

### Bỏ từng lớp con

Một phép thử khác: đặt đầu ra của đúng một lớp chú ý hoặc một MLP bằng 0 (giữ nguyên kết nối tắt) rồi đo lại. Ô dưới in
xác suất của đáp án khi bỏ từng lớp con, và số bit tăng thêm trên mỗi token của 10 câu FLORES-200 tiếng Anh.

In [ ]:
for x in inside["lens"]:
    ab, ans = x["ablate"], TOK[x["answer"]]
    rest = [a for kind in ("attn", "mlp") for a in ab[kind][1:]]  # 22 lớp con ngoài khối 0
    keep = sum(a["top"] == x["answer"] for a in rest)
    print(
        f"“{ans}” {pct(x['p'], 2)} · bỏ chú ý khối 0: {pct(ab['attn'][0]['p'], 2)} · bỏ MLP khối 0: {pct(ab['mlp'][0]['p'], 2)}"
        f" · bỏ MLP khối 11: {pct(ab['mlp'][11]['p'], 2)} · bỏ 1 trong {len(rest)} lớp con khác: vẫn đứng đầu {keep}/{len(rest)}"
    )
AF = inside["ablateFlores"]
print(f"FLORES-en ({AF['sentences']} câu): {vn(AF['base'], 2)} bit/token khi đủ mọi lớp con")
print(f"   bỏ khối 0: +{vn(AF['attn'][0], 2)} (chú ý), +{vn(AF['mlp'][0], 2)} (MLP)")
worst_kind, worst_l = max(((k_, l) for k_ in ("attn", "mlp") for l in range(1, 12)), key=lambda kl: AF[kl[0]][kl[1]])
print(
    f"   bỏ một lớp con khác: thêm tối đa {vn(AF[worst_kind][worst_l], 2)} bit"
    f" ({'chú ý' if worst_kind == 'attn' else 'MLP'} khối {worst_l})"
)

fig, ax = plt.subplots(figsize=(7.6, 3.4))
ax.plot(range(12), AF["attn"], color=BLUE, marker="o", label="bỏ lớp chú ý")
ax.plot(range(12), AF["mlp"], color=ORANGE, marker="s", ls="--", label="bỏ MLP")
ax.set(yscale="log", xlabel="khối", ylabel="bit/token tăng thêm", xticks=range(12))
ax.set_title("GPT-2 small, FLORES-200 tiếng Anh: bỏ từng lớp con", fontsize=11)
ax.legend(frameon=False)
vn_axes(fig)
plt.show()

Bỏ chú ý hoặc MLP của **khối 0** thì mô hình sụp: mỗi token tốn thêm 3–5 bit, và đáp án rơi xuống dưới 1%. Bỏ bất kỳ lớp
con nào khác chỉ thêm tối đa nửa bit, và đáp án thường vẫn đứng đầu. Như ResNet ở Bài 13, các khối cùng ghi vào một dòng
dư, nên mất một khối không làm hỏng cả mạng. Đáng ngạc nhiên hơn: bỏ MLP của khối cuối còn làm *tăng* xác suất của cả
ba đáp án (6,38% → 12,25% với “ Paris”), dù trên FLORES chính MLP này là lớp con quan trọng nhất ngoài khối 0. Nó có vẻ
làm phân phối bớt chắc chắn: điều đó giúp mất mát trung bình trên văn bản thật, nhưng không giúp riêng các câu này.

### 144 đầu chú ý

GPT-2 small có 12 khối × 12 đầu. Demo đo từng đầu trên một đoạn *Alice ở xứ sở thần tiên* (Carroll, 1865, phạm vi
công cộng), có `<|endoftext|>` đứng đầu: phần chú ý dồn vào token đầu tiên, vào token liền trước, và điểm quy nạp đo như
ở mục 11, trên 16 chuỗi 25 token ngẫu nhiên lặp lại hai lần. Đầu được đánh số từ 0, như trong các bài báo: “5.5” là khối 5,
đầu 5. Ô dưới vẽ ba bản đồ nhiệt 12 × 12 (hàng là khối, cột là đầu, màu đậm là điểm cao) và in các đầu đứng đầu.

In [ ]:
H = inside["heads"]
maps = {"token đầu tiên": H["first"], "token liền trước": H["prev"], "điểm quy nạp": H["induction"]}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (name, M) in zip(axes, maps.items()):
    im = ax.imshow(np.array(M), cmap=SEQ, vmin=0, vmax=1)
    ax.set(title=name, xlabel="đầu", xticks=range(0, 12, 2), yticks=range(0, 12, 2))
axes[0].set_ylabel("khối")
fig.colorbar(im, ax=axes, fraction=0.02)
fig.suptitle("GPT-2 small: mỗi ô là một đầu chú ý", fontsize=11)
vn_axes(fig)
plt.show()
for name, M in maps.items():
    M = np.array(M)
    order = np.argsort(-M.ravel(), kind="stable")[:4]
    print(f"{name:<17}: " + " · ".join(f"{i // 12}.{i % 12} {pct(M.ravel()[i], 1)}" for i in order))
first = np.array(H["first"])
print(f"{(first > 0.5).sum()}/144 đầu dồn hơn nửa sự chú ý vào token đầu tiên (bể chú ý)")
print(f"trung bình theo khối: {' · '.join(pct(x, 0) for x in first.mean(1))}")

Ba nhóm đầu hiện ra rõ, và không nhóm nào được giao việc trước:
- **Bể chú ý** (attention sink, Xiao và cộng sự, 2023): khoảng một nửa số đầu, nhất là ở các khối giữa và cuối, dồn
  phần lớn sự chú ý vào token đầu tiên. Softmax luôn phải chia đủ 100%, nên khi một đầu không cần lấy gì, nó cần một chỗ
  “đỗ” vô hại, và token đầu (thấy được từ mọi vị trí) thành chỗ đó.
- **Đầu nhìn token liền trước:** đầu 4.11 đặt gần như toàn bộ sự chú ý vào token liền trước, một phép dịch vị trí gần
  hoàn hảo, giống đầu ở khối 0 của mô hình chép tí hon ở mục 11.
- **Đầu quy nạp:** 5.5, 7.10, 6.9, 5.1 nằm ở các khối sau đầu nhìn token liền trước, đúng thứ tự mà mạch hai bước đòi hỏi.

Phép thử chuỗi lặp cho thấy chúng làm gì:

In [ ]:
R = H["repeat"]
print(
    f"{R['seqs']} chuỗi {R['n']} token ngẫu nhiên, lặp lại 2 lần (đoán đều trên 50.257 token: {vn(math.log2(50257), 2)} bit)"
)
print(f"   lần 1: {vn(R['first'], 2)} bit/token, đúng hạng 1 {pct(R['top1First'], 1)}")
print(f"   lần 2: {vn(R['second'], 2)} bit/token, đúng hạng 1 {pct(R['top1Second'], 1)}")
for g in R["groups"]:
    names = ", ".join(f"{a}.{b}" for a, b in g["induction"])
    print(
        f"   bỏ {g['k']:>2} đầu quy nạp mạnh nhất ({names}): lần 2 {vn(g['bitsInduction'], 2)} bit"
        f" · bỏ {g['k']} đầu ngẫu nhiên: {vn(g['bitsRandom'], 2)}"
    )

Lần đầu, chuỗi ngẫu nhiên tốn gần 19 bit mỗi token, *tệ hơn* cả đoán đều (15,62 bit), vì GPT-2 chờ đợi văn bản có nghĩa.
Lần lặp lại chỉ còn khoảng nửa bit và đoán đúng hạng 1 ở 99% vị trí: mô hình chép từ đoạn trước *ngay trong câu lệnh*,
không học thêm gì (đây là một dạng đơn giản của học trong ngữ cảnh). Bỏ 5 đầu quy nạp mạnh nhất thì lần lặp lên hơn
7 bit; bỏ 5 đầu ngẫu nhiên thì gần như không đổi. Bỏ 2 đầu thì chưa đủ: các đầu quy nạp khác bù vào.

Một chi tiết đáng để ý: 7.2, 5.1, 6.9 và 7.10 vừa là đầu quy nạp mạnh vừa nằm trong nhóm dồn vào token đầu tiên. Trên
đoạn văn Alice không có gì lặp lại để chép, nên chúng “đỗ” ở `<|endoftext|>`. Cùng một đầu, việc nó làm tùy thuộc vào
đầu vào.

### Phân phối trên cả 50.257 token

Ở mỗi bước sinh, GPT-2 cho 50.257 logit. Tệp của demo lưu chính xác 200 logit lớn nhất, còn 50.057 logit còn lại được
gom vào các ngăn rộng 0,01 (mỗi token lấy logit ở giữa ngăn). Từ đó ta dựng lại được cả phân phối ở mọi nhiệt độ. Ô dưới
viết lại phép tính của demo, rồi so với các số mà `inside.py` tính chính xác trên đủ 50.257 logit.

In [ ]:
def classes(pr):
    """Logit theo thứ tự giảm dần và số token có logit đó: 200 token đầu (mỗi token một lớp), rồi các ngăn của phần đuôi."""
    tail = pr["tail"]
    z_tail = tail["lo"] + (np.array(tail["bins"][::-1]) + 0.5) * tail["bw"]
    z = np.concatenate([[zi for _, zi in pr["head"]], z_tail])
    return z, np.concatenate([np.ones(len(pr["head"])), tail["counts"][::-1]])


def dist_stats(pr, T):
    z, cnt = classes(pr)
    w = np.exp((z - z[0]) / T)
    p = w / (cnt * w).sum()  # xác suất của MỘT token trong mỗi lớp
    cum = np.cumsum(cnt * p)
    i = int(np.searchsorted(cum, 0.9))  # lớp mà tổng xác suất vượt 0,9
    nucleus = int(cnt[:i].sum() + math.ceil((0.9 - (cum[i - 1] if i else 0)) / p[i] - 1e-12))
    entropy = -(cnt * p * np.log2(p)).sum()
    return {"top1": p[0], "top50": p[:50].sum(), "entropy": entropy, "nucleus90": nucleus}


for pr in samp["prompts"][:1] + samp["prompts"][-1:]:
    n_bins = len(pr["tail"]["bins"])
    print(
        f"“{pr['text']}”: {len(pr['head'])} logit chính xác + {vn(pr['tail']['n'], 0)} logit trong {vn(n_bins, 0)} ngăn"
    )
    for ex in pr["exact"]:
        st = dist_stats(pr, ex["T"])
        print(
            f"  T = {vn(ex['T'], 1)}: token đầu {pct(st['top1'], 1):>6} · 50 token đầu {pct(st['top50'], 1):>6} ·"
            f" entropy {vn(st['entropy'], 2):>5} bit · top-p 0,9 giữ {vn(st['nucleus90'], 0):>6} token"
            f"   (chính xác: {pct(ex['top1'], 1)} · {pct(ex['top50'], 1)} · {vn(ex['entropy'], 2)} · {vn(ex['nucleus90'], 0)})"
        )
        assert abs(st["top50"] / ex["top50"] - 1) < 2e-3 and abs(st["entropy"] - ex["entropy"]) < 0.01

Phân phối dựng lại khớp với số chính xác đến chữ số cuối được in (số token mà top-p giữ ở $T$ cao lệch 1, vì các token
trong một ngăn lấy chung một logit). Hai câu lệnh cho thấy hai điều:
- **Nhiệt độ không đổi thứ tự**, chỉ đổi độ nhọn: ở $T = 0{,}5$, 50 token đầu gom gần hết xác suất; ở $T = 2$, 50 token
  đó chỉ còn 16,1% (câu tiếng Việt: 12,7%), và top-p 0,9 phải giữ hàng nghìn token. $T$ lớn không làm mô hình “sáng tạo”
  hơn; nó chỉ cho phần đuôi dài của những token gần như vô nghĩa nhiều cơ hội được rút hơn.
- **Top-p tự co giãn** theo độ chắc chắn của mô hình. Ở $T = 1$, câu tiếng Anh cần 294 token để gom đủ 0,9 xác suất, câu
  tiếng Việt cần 225 token, dù token đứng đầu của câu tiếng Việt chỉ là một mảnh byte. Còn top-k luôn giữ đúng $k$ token.

Hình dưới vẽ 10 token đứng đầu sau câu Tháp Eiffel ở ba nhiệt độ (thanh đặc, gạch chéo và chấm). Thứ tự các token giống
hệt nhau ở cả ba nhiệt độ; chỉ chiều dài các thanh thay đổi.

In [ ]:
pr = next(q for q in samp["prompts"] if q["name"] == "eiffel")
z, cnt = classes(pr)
names = [TOK[i] for i, _ in pr["head"][:10]]
fig, ax = plt.subplots(figsize=(8, 4))
for j, (T, c, hatch) in enumerate(((0.5, BLUE, ""), (1.0, ORANGE, "///"), (2.0, GREEN, ".."))):
    w = np.exp((z - z[0]) / T)
    p10 = (w / (cnt * w).sum())[:10]
    ax.barh(
        np.arange(10) + (j - 1) * 0.27,
        p10,
        height=0.27,
        color=c,
        hatch=hatch,
        edgecolor="white",
        label=f"T = {vn(T, 1)}",
    )
ax.set_yticks(range(10), [f"“{n_}”" for n_ in names])
ax.invert_yaxis()
ax.set(xlabel="xác suất")
ax.set_title(f"“{pr['text']} …”: 10 token đầu ở ba nhiệt độ", fontsize=11)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
ax.legend(frameon=False)
plt.tight_layout()
vn_axes(fig)
plt.show()

Cuối cùng là các đoạn GPT-2 thật sự viết tiếp (40 token, seed 0), với cách rút khác nhau. Cột cuối là tỉ lệ cụm 4 token đã
gặp trước đó trong chính đoạn văn: một thước đo đơn giản cho việc lặp lại.

In [ ]:
for name in ("ex0", "eiffel"):
    pr = next(q for q in samp["prompts"] if q["name"] == name)
    print(f"“{pr['text']}”")
    for c in pr["cont"]:
        if c["seed"] == 0 and c["setting"] in ("greedy", "T0.7", "T1k40", "T2"):
            label = {"greedy": "tham lam", "T0.7": "T = 0,7", "T1k40": "T = 1, top-k 40", "T2": "T = 2"}[c["setting"]]
            text_ = gpt_text(c["ids"]).replace("\n", " ⏎ ")
            print(f"  {label:<16} lặp {pct(c['rep4'], 0):>4} │ {text_[:110]}")

Sinh tham lam rơi vào vòng lặp: “The visualization is based on the following principles” rồi lặp lại, và câu Tháp Eiffel
lặp lại nguyên văn câu lệnh, 30–56% cụm 4 token là cụm cũ. $T = 2$ không lặp nhưng ra những từ ghép bừa. Ở giữa,
$T = 0{,}7$ hay top-k 40 cho văn bản trôi chảy (dù không đúng sự thật: “Cologne, Germany”). Bài báo GPT-2 dùng chính
top-k 40 cho các đoạn văn mẫu của nó. Đây cũng là lý do *mô hình ngôn ngữ* chưa phải *trợ lý*: nó viết tiếp một văn bản
có vẻ hợp lý, không trả lời câu hỏi (Bài 26).

## 13 · Thử thay đổi

Các ô dưới chạy lại các hàm ở trên với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

**Bỏ từng phần của GPT nhỏ.** Giống phép thử trên GPT-2: đặt đầu ra của một lớp con bằng 0, hoặc xóa bảng vị trí $P$,
rồi đo lại mất mát kiểm định (không huấn luyện lại).

In [ ]:
def without(p, keys):
    return {k_: (np.zeros_like(a) if k_ in keys else a) for k_, a in p.items()}


print(f"đủ mọi phần: {vn(gpt_bits, 3)} bit/ký tự")
for l in range(CFG["L"]):
    print(f"bỏ chú ý khối {l}: {vn(val_bits(without(params, {f'{l}.Wo', f'{l}.bo'}), CFG), 3)}", end=" · ")
    print(f"bỏ MLP khối {l}: {vn(val_bits(without(params, {f'{l}.W2', f'{l}.c2'}), CFG), 3)}")
no_pos = val_bits(without(params, {"wpe"}), CFG)
print(f"xóa bảng vị trí P: {vn(no_pos, 3)} (bigram: {vn(bigram_bits, 3)})")
assert no_pos > gpt_bits + 0.5

d_ = CFG["d"]
no_bk = {k_: a.copy() for k_, a in params.items()}
for l in range(CFG["L"]):
    no_bk[f"{l}.bqkv"][d_ : 2 * d_] = 0  # đặt b_K = 0
bits_no_bk = val_bits(no_bk, CFG)
print(f"b_K sau huấn luyện: lớn nhất {vn(np.abs(params['0.bqkv'][d_ : 2 * d_]).max(), 3)} (lúc đầu: 0)")
print(f"đặt b_K = 0: {vn(bits_no_bk, 4)} bit/ký tự, so với {vn(gpt_bits, 4)}")
assert abs(bits_no_bk - gpt_bits) < 1e-4

Như GPT-2, khối đầu tiên quan trọng nhất: bỏ chú ý hay MLP của khối 0 làm mất mát tăng khoảng 1 bit trở lên, còn bỏ một
lớp con của khối 1 chỉ thêm khoảng 0,1 bit. Xóa $P$ làm mô hình còn tệ hơn bigram: nó đã học dựa vào vị trí (ví dụ để
biết đã đi được mấy tiếng trong câu), và bây giờ mất thông tin đó.

Còn $b_K$: nó bắt đầu bằng 0, gradient của nó chỉ là nhiễu làm tròn của float32, vậy mà sau huấn luyện nó không còn
bằng 0. Adam chia gradient cho căn của trung bình bình phương *của chính gradient đó*, nên một nhiễu rất nhỏ vẫn có thể
thành một bước đáng kể. $b_K$ trôi đi, nhưng đặt nó về 0 không làm mất mát đổi chút nào (chỉ khác ở mức làm tròn):
như mục 6 đã chỉ ra, nó không thể ảnh hưởng đến đầu ra.

**Nhiệt độ và mất mát.** Chia logit cho $T$ rồi đo mất mát kiểm định ở nhiều $T$. $T$ nào cho mất mát nhỏ nhất?

In [ ]:
seq_v = np.concatenate([train_ids[-1:], val_ids])
Xv = seq_v[:8000].reshape(125, 64)
Yv = seq_v[1:8001].reshape(125, 64)
zv = forward(params, Xv, CFG)[0].astype(np.float64)
temps = [0.5, 0.7, 0.8, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0]
bits_T = [cross_entropy(zv / T, Yv)[0] / math.log(2) for T in temps]
print(" · ".join(f"T = {vn(T, 2)}: {vn(b, 3)}" for T, b in zip(temps, bits_T)))
print(f"nhỏ nhất ở T = {vn(temps[int(np.argmin(bits_T))], 2)}")

Nhỏ nhất ở $T$ gần 1: mô hình được huấn luyện để chính phân phối của nó ($T = 1$) khớp với dữ liệu. Hạ $T$ khi sinh văn
bản là đổi độ đa dạng lấy độ chắc chắn; nó không làm mô hình đoán tốt hơn.

**Câu lệnh và cách rút khác.** Đổi `start`, `T` và `k`:

In [ ]:
start = "Đầu lòng hai ả tố nga,\n"  # thử một câu của bạn (chỉ dùng các ký tự có trong Truyện Kiều)
for T, k in ((0.3, 0), (0.8, 10)):
    out_ = decode(generate(params, [stoi[c] for c in start], 150, CFG, T, k, seed=1))
    print(f"T = {vn(T, 1)}, top-k {k or 'tắt'}:\n{start}{out_}\n")

**Huấn luyện lại.** Ô dưới gọi lại `train` của mục 7 với cấu hình khác. Mặc định nó chỉ chạy 65 bước để notebook nhanh.
Đặt `steps=650` và thêm vào `etas` các giá trị 0.02, 0.03, 0.06 để tự lặp lại phép chọn $\eta$; hoặc thử `"L": 1`,
`"heads": 1`, `"d": 64`. Mỗi lần 650 bước mất cỡ thời gian của mục 7. Seed khác đổi trọng số ban đầu và thứ tự các
khung, nên kết quả lệch đi một chút; muốn so hai cấu hình, hãy chạy vài seed (`seeds=(0, 1, 2)`) và so khoảng dao động.

In [ ]:
etas, steps, seeds = (0.04,), 65, (0,)
for eta in etas:
    for seed in seeds:
        _, log_try = train({**CFG, "L": 2, "heads": 4}, steps=steps, eta=eta, seed=seed)
        print(f"η = {vn(eta, 2)}, seed {seed}, {steps} bước: kiểm định {vn(log_try['val'][-1], 3)} bit/ký tự")

Sau 65 bước, mô hình mới ở khoảng 3,5 bit, tệ hơn bigram; phải đủ 650 bước như mục 7 nó mới xuống 2,56 bit.

## 14 · Giới hạn và bước tiếp theo

**Cùng khung xương, các chi tiết đã đổi.** Các mô hình mở như Llama giữ khối của GPT-2 nhưng thay bốn chi tiết: RMSNorm
thay LayerNorm (Bài 10), MLP có cổng SwiGLU (ba ma trận thay vì hai), RoPE xoay $q$ và $k$ theo vị trí thay cho bảng
$P$ cộng vào đầu vào, và GQA để nhiều đầu truy vấn dùng chung một cặp khóa/giá trị. Ô dưới đếm Llama 3 8B từ cấu hình
đã công bố (Grattafiori và cộng sự, 2024) bằng cùng cách đếm, và so bộ nhớ đệm KV của nó với GPT-2.

In [ ]:
def count_params(L, d, heads, kv_heads, hd, ff, vocab, ctx, swiglu, tied):
    q, kv = heads * hd, kv_heads * hd
    bias = not swiglu  # GPT-2 có độ lệch và LayerNorm (g, b); Llama không có độ lệch, RMSNorm chỉ có g
    attn = L * (d * (q + 2 * kv) + q * d + (q + 2 * kv + d if bias else 0))
    mlp = L * (3 * d * ff if swiglu else 2 * d * ff + ff + d)
    norm = (2 * L + 1) * (d if swiglu else 2 * d)
    pos = 0 if swiglu else ctx * d  # Llama: RoPE, không có bảng vị trí
    return vocab * d + pos + attn + mlp + norm + (0 if tied else vocab * d)


gpt2 = {"L": 12, "d": 768, "heads": 12, "kv_heads": 12, "hd": 64, "ff": 3072, "vocab": 50257, "ctx": 1024}
llama3 = {"L": 32, "d": 4096, "heads": 32, "kv_heads": 8, "hd": 128, "ff": 14336, "vocab": 128256, "ctx": 8192}
n_gpt2, n_llama = count_params(**gpt2, swiglu=False, tied=True), count_params(**llama3, swiglu=True, tied=False)
for name, c, n_ in (("GPT-2 small", gpt2, n_gpt2), ("Llama 3 8B", llama3, n_llama)):
    kv = 2 * c["L"] * c["kv_heads"] * c["hd"] * 2  # khóa + giá trị, fp16
    mha = 2 * c["L"] * c["heads"] * c["hd"] * 2  # nếu mỗi đầu truy vấn có cặp K, V riêng
    print(
        f"{name:<12} {vn(n_, 0):>13} tham số · {c['heads']} đầu truy vấn, {c['kv_heads']} đầu KV · KV {vn(kv / 1024, 0)} KiB"
        f"/token (không GQA: {vn(mha / 1024, 0)}) · {vn(c['ctx'], 0)} token: {vn(kv * c['ctx'] / 2**30, 2)} GiB"
        f" (không GQA: {vn(mha * c['ctx'] / 2**30, 2)})"
    )
assert n_gpt2 == 124439808 and n_llama == 8030261248  # khớp số đếm trên trọng số thật (GPT-2) và Hugging Face (Llama)

Llama 3 8B lớn gấp khoảng 65 lần GPT-2 small. Nhờ GQA (32 đầu truy vấn, chỉ 8 đầu KV), mỗi token chiếm 128 KiB bộ nhớ đệm
thay vì 512 KiB, và một chuỗi 8.192 token chiếm 1 GiB thay vì 4 GiB.

**Những gì notebook này chưa làm:**
- GPT nhỏ chỉ đọc 64 ký tự và đã học 650 bước. Chất lượng tăng theo kích thước mô hình, lượng dữ liệu và số phép tính
  theo những quy luật đo được: [Bài 23 · Quy luật mở rộng](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/quy-luat-mo-rong/)
  đo chúng bằng chính kiểu GPT ký tự này trên Wikipedia tiếng Việt.
- Chi phí của tự chú ý tăng theo bình phương độ dài chuỗi, và bộ nhớ đệm KV tăng theo độ dài:
  [Bài 25](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rnn-tro-lai/) thay nó bằng một trạng thái cỡ cố định.
- Một mô hình ngôn ngữ chỉ viết tiếp; biến nó thành trợ lý là việc của
  [Bài 26](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/hau-huan-luyen/).

Bỏ mặt nạ và đổi token chữ thành các mảnh ảnh, ta được cùng khối Transformer đọc ảnh: Vision Transformer, bài kế tiếp.

## Tóm tắt
- Đầu vào của GPT: $h_0[t] = E[x_t] + P[t]$. Tự chú ý: $\operatorname{softmax}(QK^\top/\sqrt{d_h} + M)\,V$, trong đó mặt
  nạ $M$ gán $-\infty$ cho các vị trí phía sau; nhiều đầu chạy song song, nối lại rồi nhân $W_O$.
- Khối pre-LN: $h \leftarrow h + \operatorname{MSA}(\operatorname{LN}(h))$, rồi $h \leftarrow h + \operatorname{MLP}(\operatorname{LN}(h))$.
  Lượt ngược viết tay khớp với sai phân hữu hạn; gradient của $b_K$ luôn bằng 0.
- GPT nhỏ 72.516 tham số, 650 bước AdamW: Truyện Kiều từ 7,04 xuống 2,56 bit/ký tự (bigram 3,14).
- Có mặt nạ, đổi một token phía sau không đổi chút nào các dự đoán phía trước, cả ở GPT nhỏ lẫn GPT-2 (Δ = 0). Nhờ vậy
  một lượt huấn luyện được mọi vị trí.
- Bộ nhớ đệm KV cho đúng kết quả của lượt đầy đủ. GPT-2: 36 KiB/token; sinh 1.018 token rẻ hơn 352 lần so với tính lại.
- GPT-2 small có 124.439.808 tham số, MLP chiếm 2/3 mỗi khối. Đầu quy nạp cần hai khối: mô hình 2 khối chép được đoạn lặp,
  mô hình 1 khối thì không. Trong GPT-2, bỏ 5 đầu quy nạp đẩy lần lặp từ 0,57 lên 7,71 bit.
- Nhiệt độ không đổi thứ tự các token, chỉ đổi độ nhọn; mất mát kiểm định nhỏ nhất ở $T = 1$. Tham lam dễ lặp lại.

## Câu hỏi ôn tập
1. Vì sao mất mát ban đầu của GPT nhỏ đúng bằng $\log_2 132$ bit, dù các embedding và ma trận chú ý là ngẫu nhiên?
2. Sửa token thứ 20 của câu. Vì sao dự đoán ở 19 vị trí đầu không đổi? Vì sao mặt nạ vẫn cần khi huấn luyện, khi mô hình
   đã có cả câu?
3. Vì sao độ lệch của khóa $b_K$ luôn có gradient bằng 0?
4. Mô hình một khối có 2 đầu chú ý. Vì sao nó không chép được đoạn lặp, còn mô hình hai khối thì được?
5. Một token của GPT-2 chiếm bao nhiêu byte trong bộ nhớ đệm KV (fp16)? Bộ nhớ đệm có làm thay đổi văn bản được sinh không?
6. Tăng nhiệt độ $T$ có đổi token đứng đầu không? Vì sao mất mát kiểm định nhỏ nhất ở $T = 1$?

<details><summary>Gợi ý đáp án</summary>

1. Lớp đầu ra $W_h$ và $b_h$ bắt đầu bằng 0, nên mọi logit bằng 0 và softmax chia đều cho 132 ký tự.
2. Với mặt nạ, vị trí $t$ chỉ đọc các vị trí $j \le t$, nên token thứ 20 không đi vào phép tính của vị trí nào đứng trước.
   Khi huấn luyện, mỗi vị trí phải đoán token kế tiếp; không có mặt nạ, nó nhìn thẳng vào đáp án (đúng là token kế tiếp
   nằm trong đầu vào), học cách “chép đáp án”, và vô dụng khi sinh văn bản, lúc tương lai chưa có.
3. $b_K$ cộng cùng một vectơ vào mọi khóa, nên mọi điểm trong một hàng tăng cùng một lượng $q_t \cdot b_K$; softmax
   không đổi khi mọi đầu vào tăng như nhau.
4. Để chép, vị trí chứa [B] phải mang sẵn thông tin “đứng trước tôi là [A]”. Thông tin đó chỉ có sau một lớp chú ý (đầu
   nhìn token liền trước), nên cần lớp chú ý thứ hai để tìm nó. Một khối chỉ biết những ký hiệu nào đã xuất hiện.
5. $2 \times 12 \times 768 \times 2 = 36.864$ byte (36 KiB). Không: nó cho đúng logit của lượt đầy đủ (chỉ lệch ở mức
   làm tròn), vì khóa và giá trị của vị trí cũ không phụ thuộc token mới.
6. Không: chia mọi logit cho cùng một số dương không đổi thứ tự. Mô hình được huấn luyện để chính phân phối ở $T = 1$
   khớp với dữ liệu; $T$ khác làm nó quá chắc hoặc quá phân vân.
</details>

## Tìm hiểu thêm
- A. Vaswani và cộng sự (2017), [Attention Is All You Need](https://arxiv.org/abs/1706.03762).
- A. Radford và cộng sự (2019),
  [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
  (GPT-2), và [README của OpenAI](https://github.com/openai/gpt-2) với lời đính chính 117M → 124M.
- A. Karpathy, [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY) và
  [nanoGPT](https://github.com/karpathy/nanoGPT): cùng một GPT ký tự, viết bằng PyTorch.
- J. Alammar, [The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/).
- N. Elhage và cộng sự (2021), [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html)
  và C. Olsson và cộng sự (2022), [In-context Learning and Induction Heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html).
- nostalgebraist (2020), [interpreting GPT: the logit lens](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens);
  N. Belrose và cộng sự (2023), [tuned lens](https://arxiv.org/abs/2303.08112).
- G. Xiao và cộng sự (2023), [Efficient Streaming Language Models with Attention Sinks](https://arxiv.org/abs/2309.17453).
- A. Holtzman và cộng sự (2019), [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751) (top-p).
- A. Haviv và cộng sự (2022), [Transformer Language Models without Positional Encodings Still Learn Positional
  Information](https://arxiv.org/abs/2203.16634).
- J. Kaplan và cộng sự (2020), [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361) (≈ 2N FLOP mỗi token).
- Polo Club, [Transformer Explainer](https://poloclub.github.io/transformer-explainer): GPT-2 chạy trong trình duyệt, nguồn của reel.

**Bài tiếp theo:** [Bài 18 · Vision Transformer (ViT)](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vision-transformer/):
cùng khối Transformer, không mặt nạ, đọc ảnh như một chuỗi mảnh.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")